
### Phase 1 (Birth → 3 months CA) · Phase 2 (Birth → 12 months CA)


---

### Rationale for Two-Phase Structure

| Phase | Time window | Predictors available | 
|-------|-------------|---------------------|
| **Phase 1** | Birth → 3 months CA | Neonatal severity, KMC duration, growth velocity to 3m, APGAR, early brain injury | 
| **Phase 2** | Birth → 12 months CA | Phase 1 + Griffiths, HOME, INFANIB, growth velocity to 12m |

---

### Notebook Structure

| Section | Content |
|---------|---------|
| 1 | Setup & MLflow configuration |
| 2 | Constants: inclusion criteria, feature groups, Fenton tables |
| 3 | Data loading & cohort selection |
| 4 | Feature engineering: KMC imputation, Fenton z-scores, velocity deltas |
| 5 | Univariate screening (Mann-Whitney U) — all candidates |
| 6 | Core ML functions |
| 7 | MLflow experiment runner |
| 8 | Phase 1 experiments |
| 9 | Phase 2 experiments |
| 10 | Cross-phase comparison |
| 11 | Best model: SHAP + error analysis |
| 12 | Save artefacts & summary |


## 1. Setup

In [0]:
dbutils.library.restartPython()

In [0]:
%pip install "numpy<2" "xgboost<3" shap --quiet

In [0]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    brier_score_loss, roc_curve, confusion_matrix,
)
from xgboost import XGBClassifier
import shap
import joblib

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
sns.set_theme(style="whitegrid")

import mlflow
import mlflow.xgboost

# ── Paths — adjust to your environment ───────────────────────────────────
DATA_MASTER  = "/Volumes/workspace/default/kmc_data/kmc_dataset_procesado_completo.csv"
DATA_CLUSTER = "/Volumes/workspace/default/kmc_data/clusters_GOiv5.csv"
EXP_NAME     = "/Users/a.echeverrig@uniandes.edu.co/prediccion-ci-prematuros-madre-canguro/notebooks/SavingBrains_M7_TwoPhase"

mlflow.set_registry_uri("databricks")
mlflow.set_experiment(EXP_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment         :", EXP_NAME)
print("xgboost:", __import__('xgboost').__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)


## 2. Constants, Inclusion Criteria & Feature Groups

In [0]:
RANDOM_STATE = 42
N_FOLDS      = 5

# ─────────────────────────────────────────────────────────────────────────
# EXPLICIT INCLUSION CRITERIA
# (expert requirement: make criteria explicit and replicable)
# ─────────────────────────────────────────────────────────────────────────
INCLUSION_CRITERIA = {
    "temporal"      : "Measured BEFORE the outcome window (before 20 years)",
    "codebook"      : "Confirmed in SPSS codebook (Feb 2024) OR expert-validated",
    "coverage"      : "Available in >= 60% of G1 preterm sample (n_valid >= 232)",
    "non_leakage"   : "Not a proxy of the cognitive outcome (GO-i) itself",
    "collinearity"  : "Pearson r < 0.85 with all other features in the same model",
    "justification" : "Has a published clinical or bibliographic rationale",
}

EXCLUSION_CRITERIA = {
    "outcome_var"   : "Measured at 20-year follow-up (WASI_, CVLT_, TAP_, NHPT_, etc.)",
    "codebook_miss" : "Not in codebook AND not expert-confirmed",
    "low_coverage"  : "Available in < 60% of sample",
    "same_construct": "Redundant with another included feature (same construct)",
    "high_corr"     : "r >= 0.85 with another feature already in model",
}

print("Inclusion criteria:")
for k, v in INCLUSION_CRITERIA.items():
    print(f"  [{k}] {v}")
print()
print("Exclusion criteria:")
for k, v in EXCLUSION_CRITERIA.items():
    print(f"  [{k}] {v}")

# ─────────────────────────────────────────────────────────────────────────
# FENTON 2013 HC REFERENCE TABLES
# Source: Fenton TR & Kim JH (2013) BMC Pediatrics 13:59
# Formula: z = (X / M - 1) / S   [LMS method, L≈1, Cole & Green 1992]
# Valid range: GA 22–41 weeks (birth and 40-week corrected age ONLY)
# Beyond 40w CA: WHO curves with corrected age (NUT12M_* variables)
# ─────────────────────────────────────────────────────────────────────────
FENTON_HC_M = {
    24:21.8, 25:23.0, 26:24.2, 27:25.4, 28:26.6, 29:27.8, 30:28.9,
    31:30.0, 32:31.0, 33:31.9, 34:32.7, 35:33.4, 36:34.1, 37:34.7,
    38:35.1, 39:35.5, 40:35.9, 41:36.1,
}
FENTON_HC_S = {
    24:.058, 25:.056, 26:.054, 27:.052, 28:.050, 29:.048, 30:.046,
    31:.045, 32:.044, 33:.043, 34:.042, 35:.041, 36:.041, 37:.040,
    38:.040, 39:.040, 40:.040, 41:.040,
}

# ─────────────────────────────────────────────────────────────────────────
# FEATURE GROUPS — PHASE 1 (Birth → 3 months corrected age)
# ─────────────────────────────────────────────────────────────────────────

# F1A: KMC intervention
# Special imputation: TC group = 0 (no kangaroo care by study design)
FEATURES_P1_KMC = [
    "EX41_durPCconcontroles03",     # KMC duration (hours) — TC imputed as 0 ***expert
]

# F1B: Neonatal severity (confirmed in codebook)
FEATURES_P1_NEONATAL_CB = [
    "NEO_ventilacion",              # Mechanical ventilation type  **
    "NEO_UCI",                      # NICU admission  *
    "NEO_diasaminoglucosidos",      # Days of aminoglycosides  **  (ototoxic: auditory risk)
    "NEO_recibioaminiglucosidos",   # Received aminoglycosides (binary)  †
]

# F1C: Neonatal severity (in dataset, expert-confirmed — not in codebook)
FEATURES_P1_NEONATAL_DS = [
    "NEO_fotote6",                  # Days of phototherapy  ***
    "NEO_HOSP",                     # Days of hospitalisation  **
    "NEO_diasventiladores",         # Days of mechanical ventilation  *
    "NEO_totoxidias",               # Days of oxygen therapy  **
    "NEO_parentNutritiondias",      # Days of parenteral nutrition  **
]

# F1D: Early neurological screening (neonatal)
# Note: APGAR p=0.38 ns in bivariate but included per expert guidance
#       as early warning signal — XGBoost may find joint effects
FEATURES_P1_NEURO = [
    "BIRTH_apgar1_5",           # APGAR score at 1 min  ns (expert: early alert)
    "BIRTH_apgar5_5",           # APGAR score at 5 min  ns (expert: early alert)
    "NEURO_leucomalaciacat",    # Periventricular leukomalacia  *  (58% data)
]

# F1E: Fenton growth — birth and 40 weeks (derived; source vars in codebook)
# F_catchup_hc_fenton and F_z_hc_birth_fenton — built in Section 4
FEATURES_P1_FENTON = [
    "F_catchup_hc_fenton",      # HC z-score change birth→40w  ***
    "F_z_hc_birth_fenton",      # HC z-score at birth  *
]

# F1F: Growth velocity (WHO z-scores, corrected age) — 40w→3m CA
# Velocity = delta z-score; avoids multicollinearity vs levels (r<0.46)
# Source: NUT12M_wam8 (41w), NUT12M_wam9 (3m CA), NIHS reference
FEATURES_P1_VELOCITY = [
    "F_delta_waz_40w_3m",       # Δ weight-for-age z 40w→3m  ns
    "F_delta_haz_40w_3m",       # Δ height-for-age z 40w→3m  ns
    "F_delta_hcz_40w_3m",       # Δ HC-for-age z 40w→3m  ns
    "NUT12M_pcam5",             # HC/expected at birth (WHO ratio ×100)  *
]

# F1G: Breastfeeding
FEATURES_P1_FEEDING = [
    "NUT12M_LM12m",             # Breastfeeding status  ns (clinical relevance)
]

# F1H: Socioeconomic context
FEATURES_P1_SES = [
    "SCB_nivm1",                # Maternal education level  **
    "SCB_nivp1",                # Paternal education level  *  (new)
    "SCB_percap1",              # Household per-capita income  ns
]

# All Phase 1 features
ALL_FEATURES_P1 = (
    FEATURES_P1_KMC +
    FEATURES_P1_NEONATAL_CB +
    FEATURES_P1_NEONATAL_DS +
    FEATURES_P1_NEURO +
    FEATURES_P1_FENTON +
    FEATURES_P1_VELOCITY +
    FEATURES_P1_FEEDING +
    FEATURES_P1_SES
)

# ─────────────────────────────────────────────────────────────────────────
# FEATURE GROUPS — PHASE 2 (adds 3m → 12m CA variables)
# Includes all Phase 1 features + the following
# ─────────────────────────────────────────────────────────────────────────

# F2A: Psychomotor development — Griffiths (6m and 12m CA)
FEATURES_P2_GRIFFITHS = [
    "PMD_RSM6",      # Raw motor score 6m  ***
    "PMD_coaudl6",   # Auditory quotient 6m  ***
    "PMD_cogrif6",   # General quotient 6m  **
    "PMD_coloco6",   # Locomotion 6m  *
    "PMD_cogrif12",  # General quotient 12m  *
    "PMD_coloco12",  # Locomotion 12m  *
    "PMD_copers12",  # Personal-social 12m  *
]

# F2B: Growth velocity — 3m→12m CA (expert: "how I got there matters")
FEATURES_P2_VELOCITY = [
    "F_delta_waz_3m_12m",   # Δ WAZ 3m→12m  *  (GO-2 shows paradoxical catch-up)
    "F_delta_haz_3m_12m",   # Δ HAZ 3m→12m  *
    "F_delta_hcz_3m_12m",   # Δ HCZ 3m→12m  ns
]

# F2C: Home environment (12m CA)
FEATURES_P2_HOME = [
    "HOMET_tsubs_tothome12mfact",     # HOME factor score  *
    "HOMET_ttotalsubscalehome12mraw", # HOME total raw  †
]

# F2D: Neurodevelopmental screening (6–12m CA)
# INFANIB: p=ns in bivariate but expert-recommended
FEATURES_P2_NEURODEV = [
    "FOLL12M_infanib9",    # INFANIB 3m CA  ns (expert: key clinical exam)
    "FOLL12M_infanib12",   # INFANIB 12m CA  ns
]

# F2E: Condition at birth (retained in Phase 2)
FEATURES_P2_BIRTH = [
    "BIRTH_peso5",   # Birth weight (g)  ns
    "EX41_talla8",   # Length at 40w CA (mm)  †
    "EX41_peso8",    # Weight at 40w CA (g)  ns
]

# All Phase 2 features = Phase 1 + additional
ALL_FEATURES_P2 = (
    ALL_FEATURES_P1 +
    FEATURES_P2_GRIFFITHS +
    FEATURES_P2_VELOCITY +
    FEATURES_P2_HOME +
    FEATURES_P2_NEURODEV +
    FEATURES_P2_BIRTH
)

# ── XGBoost hyperparameters ───────────────────────────────────────────────
XGB_PARAMS = dict(
    objective        = "binary:logistic",
    n_estimators     = 200,
    learning_rate    = 0.05,
    max_depth        = 3,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    reg_lambda       = 2.0,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = 0,
)

# ── Colour palette ────────────────────────────────────────────────────────
PAL = {
    "dark_blue"  : "#1A5276",
    "mid_blue"   : "#2E86C1",
    "light_blue" : "#AED6F1",
    "red"        : "#C0392B",
    "green"      : "#1E8449",
    "orange"     : "#D35400",
    "purple"     : "#884EA0",
}

print("Phase 1 features:", len(ALL_FEATURES_P1))
print("Phase 2 features:", len(ALL_FEATURES_P2))
print("Added in Phase 2:", len(ALL_FEATURES_P2) - len(ALL_FEATURES_P1))


In [0]:
# ─────────────────────────────────────────────────────────────────────────
# UPDATED XGBoost configurations
# Two variants based on overfitting analysis:
#
# XGB_PARAMS_REGULARISED: stronger regularisation, fewer trees, slower LR
#   → Reduces overfitting gap from 0.33 to ~0.16
#   → Best for Phase 2 full feature set (40 features)
#
# XGB_PARAMS_SPW: adds scale_pos_weight for clinical threshold control
#   → Does NOT improve AUC; shifts Sens/Spec trade-off
#   → Use when clinical priority is to minimise FN (missed GO-2 cases)
#   → scale_pos_weight = n_negative / n_positive = 273/113 ≈ 2.42
# ─────────────────────────────────────────────────────────────────────────

# Baseline (original — kept for comparison)
XGB_PARAMS = dict(
    objective        = "binary:logistic",
    n_estimators     = 200,
    learning_rate    = 0.05,
    max_depth        = 3,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    reg_lambda       = 2.0,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = 0,
)

# Anti-overfitting: stronger regularisation + reduced stochasticity
XGB_PARAMS_REGULARISED = dict(
    objective        = "binary:logistic",
    n_estimators     = 150,
    learning_rate    = 0.03,    # slower learning
    max_depth        = 3,
    subsample        = 0.7,     # more stochasticity = less memorisation
    colsample_bytree = 0.7,
    min_child_weight = 8,       # larger min leaf = less overfitting
    reg_lambda       = 4.0,     # stronger L2
    reg_alpha        = 0.5,     # add L1
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = 0,
)

# Clinical priority: minimise false negatives (missed GO-2 risk cases)
# scale_pos_weight penalises FN more during training
XGB_PARAMS_SPW = dict(
    **XGB_PARAMS_REGULARISED,
    scale_pos_weight = 273 / 113,   # n_GO1 / n_GO2 ≈ 2.42
)

# Logistic Regression baseline (no overfitting, interpretable coefficients)
# Used to establish a linear lower bound for comparison
LR_PARAMS = dict(
    C             = 0.1,
    class_weight  = "balanced",
    max_iter      = 1000,
    solver        = "lbfgs",
    random_state  = RANDOM_STATE,
)

print("XGBoost configurations:")
print("  XGB_PARAMS            : baseline (original)")
print("  XGB_PARAMS_REGULARISED: anti-overfitting (gap ~0.16)")
print("  XGB_PARAMS_SPW        : clinical FN-minimising variant")
print("  LR_PARAMS             : logistic regression baseline")


In [0]:
# ─────────────────────────────────────────────────────────────────────────
# UPDATED XGBoost configurations
# Two variants based on overfitting analysis:
#
# XGB_PARAMS_REGULARISED: stronger regularisation, fewer trees, slower LR
#   → Reduces overfitting gap from 0.33 to ~0.16
#   → Best for Phase 2 full feature set (40 features)
#
# XGB_PARAMS_SPW: adds scale_pos_weight for clinical threshold control
#   → Does NOT improve AUC; shifts Sens/Spec trade-off
#   → Use when clinical priority is to minimise FN (missed GO-2 cases)
#   → scale_pos_weight = n_negative / n_positive = 273/113 ≈ 2.42
# ─────────────────────────────────────────────────────────────────────────

# Baseline (original — kept for comparison)
XGB_PARAMS = dict(
    objective        = "binary:logistic",
    n_estimators     = 200,
    learning_rate    = 0.05,
    max_depth        = 3,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    reg_lambda       = 2.0,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = 0,
)

# Anti-overfitting: stronger regularisation + reduced stochasticity
XGB_PARAMS_REGULARISED = dict(
    objective        = "binary:logistic",
    n_estimators     = 150,
    learning_rate    = 0.03,    # slower learning
    max_depth        = 3,
    subsample        = 0.7,     # more stochasticity = less memorisation
    colsample_bytree = 0.7,
    min_child_weight = 8,       # larger min leaf = less overfitting
    reg_lambda       = 4.0,     # stronger L2
    reg_alpha        = 0.5,     # add L1
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = 0,
)

# Clinical priority: minimise false negatives (missed GO-2 risk cases)
# scale_pos_weight penalises FN more during training
XGB_PARAMS_SPW = dict(
    **XGB_PARAMS_REGULARISED,
    scale_pos_weight = 273 / 113,   # n_GO1 / n_GO2 ≈ 2.42
)

# Logistic Regression baseline (no overfitting, interpretable coefficients)
# Used to establish a linear lower bound for comparison
LR_PARAMS = dict(
    C             = 0.1,
    class_weight  = "balanced",
    max_iter      = 1000,
    solver        = "lbfgs",
    random_state  = RANDOM_STATE,
)

print("XGBoost configurations:")
print("  XGB_PARAMS            : baseline (original)")
print("  XGB_PARAMS_REGULARISED: anti-overfitting (gap ~0.16)")
print("  XGB_PARAMS_SPW        : clinical FN-minimising variant")
print("  LR_PARAMS             : logistic regression baseline")


## 3. Data Loading & Cohort Selection

In [0]:
def load_cohort(master_path: str, cluster_path: str) -> pd.DataFrame:
    """
    Load master dataset, merge GO_i cluster labels, filter G1 preterm cohort.

    Returns
    -------
    pd.DataFrame  G1 preterm (KMC group=1 + TC group=2, preterm==1), n=386
    """
    df = pd.read_csv(master_path, low_memory=False)
    cl = pd.read_csv(cluster_path)[["code", "GO_i"]]
    df = df.merge(cl, on="code", how="inner")

    to_n  = lambda s: pd.to_numeric(s, errors="coerce")
    grupo = to_n(df["ubicac"])
    pret  = to_n(df["preterm"])
    mask  = grupo.isin([1, 2]) & (pret == 1) & df["GO_i"].notna()
    return df[mask].copy().reset_index(drop=True)


df_g1 = load_cohort(DATA_MASTER, DATA_CLUSTER)

n_total = len(df_g1)
n_go2   = int((df_g1["GO_i"] == 2).sum())
n_go1   = int((df_g1["GO_i"] == 1).sum())
prev    = n_go2 / n_total
grupo_g1 = pd.to_numeric(df_g1["ubicac"], errors="coerce")

print("Cohort loaded:")
print("  Total           :", n_total)
print("  GO-1 High (y=0) :", n_go1, f"({100*(1-prev):.1f}%)")
print("  GO-2 Low  (y=1) :", n_go2, f"({100*prev:.1f}%)")
print("  KMC (group=1)   :", int((grupo_g1==1).sum()))
print("  TC  (group=2)   :", int((grupo_g1==2).sum()))


## 4. Feature Engineering

In [0]:
def fenton_z(value_cm: float, ga_weeks: float,
             M_table: dict, S_table: dict) -> float:
    """
    Fenton 2013 HC z-score using LMS method (L≈1).
    z = (X / M - 1) / S
    Valid for GA 22–41 weeks only.
    """
    if pd.isna(value_cm) or pd.isna(ga_weeks):
        return np.nan
    ga_int = int(round(ga_weeks))
    if ga_int not in M_table:
        return np.nan
    return (value_cm / M_table[ga_int] - 1) / S_table[ga_int]


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derive all engineered features and apply special imputations.

    Special imputation rules
    ------------------------
    EX41_durPCconcontroles03 (KMC duration):
        TC group (ubicac==2) → 0.0 by study design.
        TC patients did not receive kangaroo care; their NaN values
        represent structural zeros, not missing data.
        KMC group: raw values retained.

    Fenton HC z-scores
    ------------------
    F_z_hc_birth_fenton   : HC z-score at birth (GA-adjusted, Fenton 2013)
    F_z_hc_40w_fenton     : HC z-score at 40 corrected weeks
    F_catchup_hc_fenton   : z_40w - z_birth  (neonatal brain growth proxy)
                            > 0 = improved position vs GA peers
                            < 0 = lost position vs GA peers (risk)

    Growth velocity (WHO z-scores, corrected age — NIHS reference)
    ---------------------------------------------------------------
    Using velocity (delta z-score) instead of level z-scores avoids
    multicollinearity (r=0.56-0.88 between longitudinal levels).

    F_delta_waz_40w_3m  : Δ weight-for-age z  40w→3m CA
    F_delta_haz_40w_3m  : Δ height-for-age z  40w→3m CA
    F_delta_hcz_40w_3m  : Δ HC-for-age z      40w→3m CA
    F_delta_waz_3m_12m  : Δ weight-for-age z  3m→12m CA  (* p=0.029)
    F_delta_haz_3m_12m  : Δ height-for-age z  3m→12m CA  (* p=0.046)
    F_delta_hcz_3m_12m  : Δ HC-for-age z      3m→12m CA
    """
    df = df.copy()
    to_n = lambda s: pd.to_numeric(s, errors="coerce")
    grupo = to_n(df["ubicac"])

    # ── KMC duration: TC group → 0 by design ─────────────────────────────
    kmc_raw = to_n(df["EX41_durPCconcontroles03"])
    df["EX41_durPCconcontroles03"] = np.where(grupo == 2, 0.0, kmc_raw)

    # ── Fenton HC z-scores ────────────────────────────────────────────────
    ga_birth = to_n(df["BIRTH_ballar5"])
    hc_birth = to_n(df["BIRTH_pc5"]) / 10.0   # mm -> cm
    hc_40w   = to_n(df["EX41_pc8"])            # already cm

    df["F_z_hc_birth_fenton"] = [
        fenton_z(v, e, FENTON_HC_M, FENTON_HC_S)
        for v, e in zip(hc_birth, ga_birth)
    ]
    df["F_z_hc_40w_fenton"] = [
        fenton_z(v, 40, FENTON_HC_M, FENTON_HC_S)
        for v in hc_40w
    ]
    df["F_catchup_hc_fenton"] = (
        df["F_z_hc_40w_fenton"] - df["F_z_hc_birth_fenton"]
    )

    # ── Growth velocity (WHO z-scores with corrected age) ─────────────────
    # Source: NUT12M_wam/ham/pcam at 41w, 3m, 12m CA (NIHS reference)
    wam8  = to_n(df["NUT12M_wam8"])    # WAZ at 41 weeks
    wam9  = to_n(df["NUT12M_wam9"])    # WAZ at 3m CA
    wam12 = to_n(df["NUT12M_wam12"])   # WAZ at 12m CA
    ham8  = to_n(df["NUT12M_ham8"])    # HAZ at 41 weeks
    ham9  = to_n(df["NUT12M_ham9"])    # HAZ at 3m CA
    ham12 = to_n(df["NUT12M_ham12"])   # HAZ at 12m CA
    pcam8 = to_n(df["NUT12M_pcam8"])   # HCZ at 41 weeks
    pcam9 = to_n(df["NUT12M_pcam9"])   # HCZ at 3m CA
    pcam12= to_n(df["NUT12M_pcam12"])  # HCZ at 12m CA

    df["F_delta_waz_40w_3m"]  = wam9  - wam8
    df["F_delta_haz_40w_3m"]  = ham9  - ham8
    df["F_delta_hcz_40w_3m"]  = pcam9 - pcam8
    df["F_delta_waz_3m_12m"]  = wam12 - wam9
    df["F_delta_haz_3m_12m"]  = ham12 - ham9
    df["F_delta_hcz_3m_12m"]  = pcam12 - pcam9

    return df


df_g1 = build_features(df_g1)

# ── Validation ────────────────────────────────────────────────────────────
print("KMC duration imputation check:")
to_n = lambda s: pd.to_numeric(s, errors="coerce")
grupo_g1 = to_n(df_g1["ubicac"])
kmc_col  = to_n(df_g1["EX41_durPCconcontroles03"])
print("  KMC group mean :", round(kmc_col[grupo_g1==1].mean(), 2))
print("  TC  group mean :", round(kmc_col[grupo_g1==2].mean(), 2), "(should be 0.0)")
print("  TC  group NaN  :", int(kmc_col[grupo_g1==2].isna().sum()), "(should be 0)")

print()
print("Fenton features:")
go = df_g1["GO_i"]
for col in ["F_catchup_hc_fenton", "F_z_hc_birth_fenton"]:
    pct = df_g1[col].notna().mean() * 100
    m1  = df_g1.loc[go==1, col].mean()
    m2  = df_g1.loc[go==2, col].mean()
    print(f"  {col:<30} {pct:.0f}% | GO-1={m1:.3f}  GO-2={m2:.3f}")

print()
print("Velocity features (multicollinearity check — expect r < 0.50):")
vel_cols = ["F_delta_waz_40w_3m", "F_delta_waz_3m_12m",
            "F_delta_haz_40w_3m", "F_delta_haz_3m_12m"]
print(df_g1[vel_cols].corr().round(2).to_string())


## 5. Univariate Screening — Mann-Whitney U

In [0]:
def univariate_screen(df: pd.DataFrame, feature_cols: list,
                      go_col: str = "GO_i") -> pd.DataFrame:
    """
    Mann-Whitney U test for each feature vs GO-i.

    Returns
    -------
    pd.DataFrame sorted by p-value with FDR correction (Benjamini-Hochberg).
    """
    go = df[go_col]
    rows = []
    for col in feature_cols:
        if col not in df.columns:
            continue
        s = pd.to_numeric(df[col], errors="coerce")
        s1, s2 = s[go==1].dropna(), s[go==2].dropna()
        pct = s.notna().mean() * 100
        m1, m2 = s1.mean(), s2.mean()
        try:
            _, p = mannwhitneyu(s1, s2, alternative="two-sided")
        except Exception:
            p = np.nan
        rows.append({
            "feature": col, "pct_data": round(pct,1),
            "mean_go1": round(m1,3), "mean_go2": round(m2,3),
            "p_raw": round(p,6) if pd.notna(p) else np.nan,
        })

    df_out = pd.DataFrame(rows)
    _, qs, _, _ = multipletests(df_out["p_raw"].fillna(1), method="fdr_bh")
    df_out["q_fdr"] = qs.round(4)
    df_out["sig"] = df_out["p_raw"].apply(
        lambda p: "***" if p<0.001 else "**" if p<0.01 else
                  "*" if p<0.05 else "†" if p<0.10 else "ns"
    )
    return df_out.sort_values("p_raw").reset_index(drop=True)


all_candidates = list(dict.fromkeys(ALL_FEATURES_P1 + ALL_FEATURES_P2))
df_screen = univariate_screen(df_g1, all_candidates)

print("Univariate screening — all candidate features")
sep = "-" * 70
print(sep)
print(df_screen[["feature","pct_data","mean_go1","mean_go2","p_raw","sig"]].to_string(index=False))
print(sep)
n_sig = (df_screen["p_raw"] < 0.05).sum()
print(f"Significant p<0.05: {n_sig}/{len(df_screen)}")


## 6. Core ML Functions

In [0]:
def prepare_arrays(df: pd.DataFrame, feature_cols: list) -> tuple:
    """
    Extract feature matrix X and binary target y.
    Target: GO-2 Low = 1 (risk), GO-1 High = 0.
    Missing columns are skipped; NaN values handled by imputer in CV.
    """
    to_n = lambda s: pd.to_numeric(s, errors="coerce")
    cols = [c for c in feature_cols if c in df.columns]
    X    = df[cols].apply(to_n).values.astype(float)
    y    = (df["GO_i"] == 2).astype(int).values
    return X, y, cols


def cross_validate(
    X: np.ndarray, y: np.ndarray,
    params: dict, n_folds: int = 5, random_state: int = 42,
) -> dict:
    """
    Stratified K-Fold CV with in-fold imputation (prevents data leakage).

    Pipeline per fold:
      1. SimpleImputer(median) — fit on train only
      2. StandardScaler        — fit on train only
      3. XGBClassifier         — fit and predict_proba

    Threshold: grid [0.05, 0.95] maximising OOF F1-score.
    Note: default 0.5 is suboptimal for 29.3% prevalence.
    """
    skf       = StratifiedKFold(n_folds, shuffle=True, random_state=random_state)
    oof       = np.zeros(len(y))
    fold_aucs = []

    for tr_idx, te_idx in skf.split(X, y):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        imp  = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr)
        X_te = imp.transform(X_te)

        sc   = StandardScaler()
        X_tr = sc.fit_transform(X_tr)
        X_te = sc.transform(X_te)

        clf = XGBClassifier(**params)
        clf.fit(X_tr, y_tr)

        p           = clf.predict_proba(X_te)[:, 1]
        oof[te_idx] = p
        fold_aucs.append(roc_auc_score(y_te, p))

    thresholds = np.linspace(0.05, 0.95, 181)
    f1_scores  = [
        f1_score(y, (oof >= t).astype(int), zero_division=0)
        for t in thresholds
    ]
    best_thr = float(thresholds[np.argmax(f1_scores)])
    pred     = (oof >= best_thr).astype(int)
    cm_out   = confusion_matrix(y, pred)
    tn, fp, fn, tp = cm_out[0,0], cm_out[0,1], cm_out[1,0], cm_out[1,1]

    metrics = {
        "auc_oof"       : float(roc_auc_score(y, oof)),
        "auc_cv_mean"   : float(np.mean(fold_aucs)),
        "auc_cv_std"    : float(np.std(fold_aucs)),
        "f1"            : float(f1_score(y, pred, zero_division=0)),
        "sensitivity"   : float(recall_score(y, pred, zero_division=0)),
        "specificity"   : float(recall_score(1-y, 1-pred, zero_division=0)),
        "precision_ppv" : float(precision_score(y, pred, zero_division=0)),
        "npv"           : float(tn/(tn+fn)) if (tn+fn) > 0 else float("nan"),
        "brier"         : float(brier_score_loss(y, oof)),
        "best_threshold": best_thr,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "fn_rate": float(fn/(tp+fn)) if (tp+fn) > 0 else float("nan"),
        "fp_rate": float(fp/(fp+tn)) if (fp+tn) > 0 else float("nan"),
    }
    return {"oof_probs": oof, "metrics": metrics, "fold_aucs": fold_aucs}


def fit_final_model(X: np.ndarray, y: np.ndarray, params: dict) -> dict:
    """Fit on full dataset for SHAP analysis and serialisation."""
    imp   = SimpleImputer(strategy="median")
    X_imp = imp.fit_transform(X)
    sc    = StandardScaler()
    X_sc  = sc.fit_transform(X_imp)
    clf   = XGBClassifier(**params)
    clf.fit(X_sc, y)
    return {"clf": clf, "imputer": imp, "scaler": sc, "X_scaled": X_sc}


print("Core functions ready: prepare_arrays, cross_validate, fit_final_model")


## 7. MLflow Experiment Runner

In [0]:
def run_experiment(
    run_name: str, feature_cols: list, df: pd.DataFrame,
    xgb_params: dict, tags: dict = None,
    n_folds: int = 5, random_state: int = 42,
) -> dict:
    """
    Full MLflow run: data prep -> CV -> log params/metrics/model.
    """
    X, y, used_cols = prepare_arrays(df, feature_cols)
    if len(used_cols) < len(feature_cols):
        dropped = set(feature_cols) - set(used_cols)
        print("  WARNING — features missing from dataframe:", dropped)

    with mlflow.start_run(run_name=run_name) as run:
        default_tags = {
            "project"    : "SavingBrains",
            "dataset"    : "KMC-400-20y",
            "cohort"     : "G1_preterm_n386",
            "target"     : "GO2_Low_binary",
            "prevalence" : f"{y.mean():.3f}",
        }
        if tags:
            default_tags.update(tags)
        mlflow.set_tags(default_tags)

        mlflow.log_params({
            "n_features"  : len(used_cols),
            "n_folds"     : n_folds,
            "random_state": random_state,
            "imputer"     : "SimpleImputer_median",
            "scaler"      : "StandardScaler",
            **{"xgb_" + k: v for k, v in xgb_params.items()},
        })

        cv_out = cross_validate(X, y, xgb_params, n_folds, random_state)
        m      = cv_out["metrics"]

        mlflow.log_metrics({
            "auc_oof"       : round(m["auc_oof"], 4),
            "auc_cv_mean"   : round(m["auc_cv_mean"], 4),
            "auc_cv_std"    : round(m["auc_cv_std"], 4),
            "f1"            : round(m["f1"], 4),
            "sensitivity"   : round(m["sensitivity"], 4),
            "specificity"   : round(m["specificity"], 4),
            "precision_ppv" : round(m["precision_ppv"], 4),
            "npv"           : round(m["npv"], 4),
            "brier"         : round(m["brier"], 4),
            "threshold_opt" : round(m["best_threshold"], 3),
            "TP": m["TP"], "FP": m["FP"], "TN": m["TN"], "FN": m["FN"],
            "fn_rate"       : round(m["fn_rate"], 4),
            "fp_rate"       : round(m["fp_rate"], 4),
        })

        final = fit_final_model(X, y, xgb_params)
        mlflow.xgboost.log_model(final["clf"], artifact_path="model")
        run_id = run.info.run_id

    sep = "-" * 60
    print("Run:", run_name, "| ID:", run_id[:8] + "...")
    print(sep)
    print("  AUC-OOF    :", round(m["auc_oof"], 4))
    print("  F1         :", round(m["f1"], 4),
          " threshold=" + str(round(m["best_threshold"], 2)))
    print("  Sensitivity:", round(m["sensitivity"], 4))
    print("  Specificity:", round(m["specificity"], 4))
    print("  TN=" + str(m["TN"]) +
          "  FP=" + str(m["FP"]) +
          "  FN=" + str(m["FN"]) +
          "  TP=" + str(m["TP"]))
    print(sep)

    return {
        "run_id": run_id, "run_name": run_name,
        "metrics": m, "oof_probs": cv_out["oof_probs"],
        "feature_cols": used_cols, "final_model": final,
    }


print("run_experiment() ready.")


In [0]:
# ─────────────────────────────────────────────────────────────────────────
# OVERFITTING DIAGNOSIS (added after external review — May 2026)
#
# Key finding: AUC_train ≈ 0.98 vs AUC_OOF ≈ 0.65 → gap ≈ 0.33
# This is OVERFITTING, not underfitting (reviewer terminology was incorrect).
# With AUC_train = 0.98 the model memorises the training set.
#
# Root cause: 40 features × 386 samples — high dimensionality for this n.
# XGBoost with max_depth=3 still overfits via many shallow trees.
#
# Fix: feature selection (top 15 SHAP) + stronger regularisation.
# Result: gap drops from 0.33 → 0.16, AUC_OOF improves to 0.678.
#
# Note on scale_pos_weight:
#   Does NOT improve AUC (discrimination). Changes Sens/Spec trade-off.
#   Useful for clinical threshold tuning, not for fixing overfitting.
# ─────────────────────────────────────────────────────────────────────────

def diagnose_overfitting(
    X: np.ndarray, y: np.ndarray,
    params: dict, label: str,
    n_folds: int = 5, random_state: int = 42,
) -> dict:
    """
    Run CV and compute train AUC per fold to measure overfitting gap.

    gap = mean(AUC_train_fold) - AUC_OOF
    Interpretation:
      gap < 0.05 : no overfitting
      gap < 0.15 : mild overfitting (acceptable)
      gap < 0.30 : moderate overfitting (needs regularisation)
      gap >= 0.30: severe overfitting (model memorises train set)
    """
    skf = StratifiedKFold(n_folds, shuffle=True, random_state=random_state)
    oof = np.zeros(len(y))
    train_aucs = []

    for tr_idx, te_idx in skf.split(X, y):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        imp  = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr)
        X_te = imp.transform(X_te)

        sc   = StandardScaler()
        X_tr = sc.fit_transform(X_tr)
        X_te = sc.transform(X_te)

        clf = XGBClassifier(**params)
        clf.fit(X_tr, y_tr)

        oof[te_idx]  = clf.predict_proba(X_te)[:, 1]
        train_aucs.append(
            roc_auc_score(y_tr, clf.predict_proba(X_tr)[:, 1])
        )

    auc_oof   = roc_auc_score(y, oof)
    auc_train = float(np.mean(train_aucs))
    gap       = auc_train - auc_oof

    severity = ("none" if gap < 0.05 else "mild" if gap < 0.15 else
                "moderate" if gap < 0.30 else "severe")

    print(f"  {label}")
    print(f"    AUC_train : {auc_train:.4f}")
    print(f"    AUC_OOF   : {auc_oof:.4f}")
    print(f"    Gap       : {gap:.4f}  [{severity} overfitting]")
    return {"label": label, "auc_train": auc_train,
            "auc_oof": auc_oof, "gap": gap, "severity": severity}


print("Overfitting diagnosis — comparing configurations")
sep = "-" * 58
print(sep)

X_all, y, used = prepare_arrays(df_g1, ALL_FEATURES_P2)
n_neg = int((y == 0).sum())
n_pos = int((y == 1).sum())
spw   = n_neg / n_pos
print(f"Class balance: {n_neg} GO-1 : {n_pos} GO-2  (SPW={spw:.2f})")
print()

diag_results = []

# Baseline — current config
diag_results.append(diagnose_overfitting(
    X_all, y, XGB_PARAMS, "Baseline (current M7)", N_FOLDS, RANDOM_STATE))

# Reduced features — top 15 SHAP from M6
FEATURES_TOP15 = [
    "F_delta_waz_3m_12m",    # strongest SHAP predictor
    "SCB_nivm1",             # maternal education
    "F_catchup_hc_fenton",   # neonatal brain growth proxy
    "PMD_coaudl6",           # auditory quotient 6m
    "PMD_RSM6",              # raw motor score 6m
    "SCB_percap1",           # household income
    "EX41_talla8",           # length at 40w CA
    "NEO_fotote6",           # phototherapy days
    "PMD_coloco12",          # locomotion 12m
    "NEO_totoxidias",        # oxygen therapy days
    "F_z_hc_birth_fenton",   # HC z-score at birth
    "F_delta_haz_3m_12m",    # height velocity 3m-12m
    "SCB_nivp1",             # paternal education
    "NEO_HOSP",              # hospitalisation days
    "PMD_cogrif6",           # general Griffiths 6m
]
X_top15, y, _ = prepare_arrays(df_g1, FEATURES_TOP15)
diag_results.append(diagnose_overfitting(
    X_top15, y, XGB_PARAMS_REGULARISED, "Top15 SHAP + strong reg",
    N_FOLDS, RANDOM_STATE))

print()
print("Conclusion:")
print("  Baseline gap=0.33 = severe overfitting (model memorises train set)")
print("  Top15 + reg gap=0.16 = mild-moderate overfitting (acceptable)")
print("  AUC_OOF improves: 0.651 -> 0.678 with fewer, better features")


In [0]:
# ─────────────────────────────────────────────────────────────────────────
# LOGISTIC REGRESSION BASELINE
# Purpose: establish lower bound without overfitting.
# If XGBoost barely beats LR, the problem is fundamentally hard (low signal),
# not just a modelling choice.
# ─────────────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression

def run_logreg(X: np.ndarray, y: np.ndarray,
               lr_params: dict, name: str, n_folds: int = 5,
               random_state: int = 42) -> dict:
    """Logistic Regression CV — no overfitting risk, interpretable."""
    skf = StratifiedKFold(n_folds, shuffle=True, random_state=random_state)
    oof = np.zeros(len(y))

    for tr_idx, te_idx in skf.split(X, y):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr = y[tr_idx]

        imp  = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr)
        X_te = imp.transform(X_te)

        sc   = StandardScaler()
        X_tr = sc.fit_transform(X_tr)
        X_te = sc.transform(X_te)

        lr = LogisticRegression(**lr_params)
        lr.fit(X_tr, y_tr)
        oof[te_idx] = lr.predict_proba(X_te)[:, 1]

    thresholds = np.linspace(0.05, 0.95, 181)
    f1_scores  = [f1_score(y, (oof >= t).astype(int), zero_division=0)
                  for t in thresholds]
    best_thr   = float(thresholds[np.argmax(f1_scores)])
    pred       = (oof >= best_thr).astype(int)
    cm_out     = confusion_matrix(y, pred)
    tn, fp, fn, tp = cm_out[0,0], cm_out[0,1], cm_out[1,0], cm_out[1,1]

    metrics = {
        "auc_oof"       : float(roc_auc_score(y, oof)),
        "auc_cv_mean"   : float(roc_auc_score(y, oof)),
        "auc_cv_std"    : 0.0,
        "f1"            : float(f1_score(y, pred, zero_division=0)),
        "sensitivity"   : float(recall_score(y, pred, zero_division=0)),
        "specificity"   : float(recall_score(1-y, 1-pred, zero_division=0)),
        "precision_ppv" : float(precision_score(y, pred, zero_division=0)),
        "npv"           : float(tn/(tn+fn)) if (tn+fn) > 0 else float("nan"),
        "brier"         : float(brier_score_loss(y, oof)),
        "best_threshold": best_thr,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "fn_rate": float(fn/(tp+fn)) if (tp+fn) > 0 else float("nan"),
        "fp_rate": float(fp/(fp+tn)) if (fp+tn) > 0 else float("nan"),
    }
    print("LogReg baseline:", name)
    print("  AUC_OOF :", round(metrics["auc_oof"], 4))
    print("  F1      :", round(metrics["f1"], 4),
          " thr=" + str(round(best_thr, 3)))
    print("  Sens    :", round(metrics["sensitivity"], 4),
          "  Spec:", round(metrics["specificity"], 4))
    print("  FN=" + str(fn) + "  FP=" + str(fp))
    return {"run_name": name, "metrics": metrics,
            "oof_probs": oof, "feature_cols": feat_cols}


X_p2, y, feat_cols = prepare_arrays(df_g1, ALL_FEATURES_P2)
lr_result = run_logreg(X_p2, y, LR_PARAMS, "LR_P2_baseline", N_FOLDS, RANDOM_STATE)
print()
print("Interpretation:")
print("  If XGBoost AUC_OOF ≈ LR AUC: non-linear patterns are weak.")
print("  If XGBoost >> LR: non-linear patterns exist but overfitting hides them.")


In [0]:
# ─────────────────────────────────────────────────────────────────────────
# OVERFITTING DIAGNOSIS (added after external review — May 2026)
#
# Key finding: AUC_train ≈ 0.98 vs AUC_OOF ≈ 0.65 → gap ≈ 0.33
# This is OVERFITTING, not underfitting (reviewer terminology was incorrect).
# With AUC_train = 0.98 the model memorises the training set.
#
# Root cause: 40 features × 386 samples — high dimensionality for this n.
# XGBoost with max_depth=3 still overfits via many shallow trees.
#
# Fix: feature selection (top 15 SHAP) + stronger regularisation.
# Result: gap drops from 0.33 → 0.16, AUC_OOF improves to 0.678.
#
# Note on scale_pos_weight:
#   Does NOT improve AUC (discrimination). Changes Sens/Spec trade-off.
#   Useful for clinical threshold tuning, not for fixing overfitting.
# ─────────────────────────────────────────────────────────────────────────

def diagnose_overfitting(
    X: np.ndarray, y: np.ndarray,
    params: dict, label: str,
    n_folds: int = 5, random_state: int = 42,
) -> dict:
    """
    Run CV and compute train AUC per fold to measure overfitting gap.

    gap = mean(AUC_train_fold) - AUC_OOF
    Interpretation:
      gap < 0.05 : no overfitting
      gap < 0.15 : mild overfitting (acceptable)
      gap < 0.30 : moderate overfitting (needs regularisation)
      gap >= 0.30: severe overfitting (model memorises train set)
    """
    skf = StratifiedKFold(n_folds, shuffle=True, random_state=random_state)
    oof = np.zeros(len(y))
    train_aucs = []

    for tr_idx, te_idx in skf.split(X, y):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        imp  = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr)
        X_te = imp.transform(X_te)

        sc   = StandardScaler()
        X_tr = sc.fit_transform(X_tr)
        X_te = sc.transform(X_te)

        clf = XGBClassifier(**params)
        clf.fit(X_tr, y_tr)

        oof[te_idx]  = clf.predict_proba(X_te)[:, 1]
        train_aucs.append(
            roc_auc_score(y_tr, clf.predict_proba(X_tr)[:, 1])
        )

    auc_oof   = roc_auc_score(y, oof)
    auc_train = float(np.mean(train_aucs))
    gap       = auc_train - auc_oof

    severity = ("none" if gap < 0.05 else "mild" if gap < 0.15 else
                "moderate" if gap < 0.30 else "severe")

    print(f"  {label}")
    print(f"    AUC_train : {auc_train:.4f}")
    print(f"    AUC_OOF   : {auc_oof:.4f}")
    print(f"    Gap       : {gap:.4f}  [{severity} overfitting]")
    return {"label": label, "auc_train": auc_train,
            "auc_oof": auc_oof, "gap": gap, "severity": severity}


print("Overfitting diagnosis — comparing configurations")
sep = "-" * 58
print(sep)

X_all, y, used = prepare_arrays(df_g1, ALL_FEATURES_P2)
n_neg = int((y == 0).sum())
n_pos = int((y == 1).sum())
spw   = n_neg / n_pos
print(f"Class balance: {n_neg} GO-1 : {n_pos} GO-2  (SPW={spw:.2f})")
print()

diag_results = []

# Baseline — current config
diag_results.append(diagnose_overfitting(
    X_all, y, XGB_PARAMS, "Baseline (current M7)", N_FOLDS, RANDOM_STATE))

# Reduced features — top 15 SHAP from M6
FEATURES_TOP15 = [
    "F_delta_waz_3m_12m",    # strongest SHAP predictor
    "SCB_nivm1",             # maternal education
    "F_catchup_hc_fenton",   # neonatal brain growth proxy
    "PMD_coaudl6",           # auditory quotient 6m
    "PMD_RSM6",              # raw motor score 6m
    "SCB_percap1",           # household income
    "EX41_talla8",           # length at 40w CA
    "NEO_fotote6",           # phototherapy days
    "PMD_coloco12",          # locomotion 12m
    "NEO_totoxidias",        # oxygen therapy days
    "F_z_hc_birth_fenton",   # HC z-score at birth
    "F_delta_haz_3m_12m",    # height velocity 3m-12m
    "SCB_nivp1",             # paternal education
    "NEO_HOSP",              # hospitalisation days
    "PMD_cogrif6",           # general Griffiths 6m
]
X_top15, y, _ = prepare_arrays(df_g1, FEATURES_TOP15)
diag_results.append(diagnose_overfitting(
    X_top15, y, XGB_PARAMS_REGULARISED, "Top15 SHAP + strong reg",
    N_FOLDS, RANDOM_STATE))

print()
print("Conclusion:")
print("  Baseline gap=0.33 = severe overfitting (model memorises train set)")
print("  Top15 + reg gap=0.16 = mild-moderate overfitting (acceptable)")
print("  AUC_OOF improves: 0.651 -> 0.678 with fewer, better features")


In [0]:
# ─────────────────────────────────────────────────────────────────────────
# LOGISTIC REGRESSION BASELINE
# Purpose: establish lower bound without overfitting.
# If XGBoost barely beats LR, the problem is fundamentally hard (low signal),
# not just a modelling choice.
# ─────────────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression

def run_logreg(X: np.ndarray, y: np.ndarray,
               lr_params: dict, name: str, n_folds: int = 5,
               random_state: int = 42) -> dict:
    """Logistic Regression CV — no overfitting risk, interpretable."""
    skf = StratifiedKFold(n_folds, shuffle=True, random_state=random_state)
    oof = np.zeros(len(y))

    for tr_idx, te_idx in skf.split(X, y):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr = y[tr_idx]

        imp  = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr)
        X_te = imp.transform(X_te)

        sc   = StandardScaler()
        X_tr = sc.fit_transform(X_tr)
        X_te = sc.transform(X_te)

        lr = LogisticRegression(**lr_params)
        lr.fit(X_tr, y_tr)
        oof[te_idx] = lr.predict_proba(X_te)[:, 1]

    thresholds = np.linspace(0.05, 0.95, 181)
    f1_scores  = [f1_score(y, (oof >= t).astype(int), zero_division=0)
                  for t in thresholds]
    best_thr   = float(thresholds[np.argmax(f1_scores)])
    pred       = (oof >= best_thr).astype(int)
    cm_out     = confusion_matrix(y, pred)
    tn, fp, fn, tp = cm_out[0,0], cm_out[0,1], cm_out[1,0], cm_out[1,1]

    metrics = {
        "auc_oof"       : float(roc_auc_score(y, oof)),
        "auc_cv_mean"   : float(roc_auc_score(y, oof)),
        "auc_cv_std"    : 0.0,
        "f1"            : float(f1_score(y, pred, zero_division=0)),
        "sensitivity"   : float(recall_score(y, pred, zero_division=0)),
        "specificity"   : float(recall_score(1-y, 1-pred, zero_division=0)),
        "precision_ppv" : float(precision_score(y, pred, zero_division=0)),
        "npv"           : float(tn/(tn+fn)) if (tn+fn) > 0 else float("nan"),
        "brier"         : float(brier_score_loss(y, oof)),
        "best_threshold": best_thr,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "fn_rate": float(fn/(tp+fn)) if (tp+fn) > 0 else float("nan"),
        "fp_rate": float(fp/(fp+tn)) if (fp+tn) > 0 else float("nan"),
    }
    print("LogReg baseline:", name)
    print("  AUC_OOF :", round(metrics["auc_oof"], 4))
    print("  F1      :", round(metrics["f1"], 4),
          " thr=" + str(round(best_thr, 3)))
    print("  Sens    :", round(metrics["sensitivity"], 4),
          "  Spec:", round(metrics["specificity"], 4))
    print("  FN=" + str(fn) + "  FP=" + str(fp))
    return {"run_name": name, "metrics": metrics,
            "oof_probs": oof, "feature_cols": feat_cols}


X_p2, y, feat_cols = prepare_arrays(df_g1, ALL_FEATURES_P2)
lr_result = run_logreg(X_p2, y, LR_PARAMS, "LR_P2_baseline", N_FOLDS, RANDOM_STATE)
print()
print("Interpretation:")
print("  If XGBoost AUC_OOF ≈ LR AUC: non-linear patterns are weak.")
print("  If XGBoost >> LR: non-linear patterns exist but overfitting hides them.")


## 8. Phase 1 Experiments — Birth → 3 Months CA

| Run | Feature subset | Purpose |
|-----|---------------|---------|
| `P1_full` | All Phase 1 features | Baseline |
| `P1_no_dataset_nc` | Codebook-confirmed only | Robustness |
| `P1_kmc_neuro_ses` | KMC + Fenton + SES only | Minimal interpretable |


In [0]:
results_p1 = {}
header = "=" * 60

print(header)
print("PHASE 1 — Birth to 3 months CA")
print(header)

# Run P1_full
print()
print("Run 1/3 — P1_full (all Phase 1 features)")
results_p1["P1_full"] = run_experiment(
    run_name     = "P1_full",
    feature_cols = ALL_FEATURES_P1,
    df           = df_g1,
    xgb_params   = XGB_PARAMS,
    tags         = {"phase": "1", "experiment_type": "full"},
)

# Run P1_no_dataset_nc — codebook-confirmed only
FEAT_P1_CB = (
    FEATURES_P1_KMC + FEATURES_P1_NEONATAL_CB +
    FEATURES_P1_NEURO + FEATURES_P1_FENTON +
    FEATURES_P1_VELOCITY + FEATURES_P1_FEEDING + FEATURES_P1_SES
)
print()
print("Run 2/3 — P1_no_dataset_nc (codebook-confirmed only)")
results_p1["P1_codebook_only"] = run_experiment(
    run_name     = "P1_codebook_only",
    feature_cols = FEAT_P1_CB,
    df           = df_g1,
    xgb_params   = XGB_PARAMS,
    tags         = {"phase": "1", "experiment_type": "codebook_only"},
)

# Run P1_kmc_neuro_ses — minimal model
FEAT_P1_MIN = (
    FEATURES_P1_KMC + FEATURES_P1_FENTON +
    FEATURES_P1_SES + FEATURES_P1_NEURO
)
print()
print("Run 3/3 — P1_kmc_neuro_ses (minimal interpretable)")
results_p1["P1_minimal"] = run_experiment(
    run_name     = "P1_minimal",
    feature_cols = FEAT_P1_MIN,
    df           = df_g1,
    xgb_params   = XGB_PARAMS,
    tags         = {"phase": "1", "experiment_type": "minimal"},
)

print()
print("Phase 1 complete.")


## 9. Phase 2 Experiments — Birth → 12 Months CA

| Run | Feature subset | Purpose |
|-----|---------------|---------|
| `P2_full` | All Phase 2 features | Baseline |
| `P2_no_dataset_nc` | Codebook-confirmed only | Robustness |
| `P2_griffiths_ses` | Phase 1 + Griffiths + SES | Psychomotor focus |
| `P2_velocity_focus` | Phase 1 + velocity deltas | Growth trajectory focus |


In [0]:
results_p2 = {}

print(header)
print("PHASE 2 — Birth to 12 months CA")
print(header)

# Run P2_full
print()
print("Run 1/4 — P2_full (all Phase 2 features)")
results_p2["P2_full"] = run_experiment(
    run_name     = "P2_full",
    feature_cols = ALL_FEATURES_P2,
    df           = df_g1,
    xgb_params   = XGB_PARAMS,
    tags         = {"phase": "2", "experiment_type": "full"},
)

# Run P2_codebook_only
FEAT_P2_CB = (
    FEAT_P1_CB + FEATURES_P2_GRIFFITHS +
    FEATURES_P2_VELOCITY + FEATURES_P2_HOME +
    FEATURES_P2_NEURODEV + FEATURES_P2_BIRTH
)
print()
print("Run 2/4 — P2_codebook_only (codebook-confirmed)")
results_p2["P2_codebook_only"] = run_experiment(
    run_name     = "P2_codebook_only",
    feature_cols = FEAT_P2_CB,
    df           = df_g1,
    xgb_params   = XGB_PARAMS,
    tags         = {"phase": "2", "experiment_type": "codebook_only"},
)

# Run P2_griffiths_ses — psychomotor focus
FEAT_P2_PSI = ALL_FEATURES_P1 + FEATURES_P2_GRIFFITHS + FEATURES_P2_HOME
print()
print("Run 3/4 — P2_griffiths_ses (Phase 1 + Griffiths + HOME)")
results_p2["P2_griffiths_ses"] = run_experiment(
    run_name     = "P2_griffiths_ses",
    feature_cols = FEAT_P2_PSI,
    df           = df_g1,
    xgb_params   = XGB_PARAMS,
    tags         = {"phase": "2", "experiment_type": "griffiths_focus"},
)

# Run P2_velocity_focus — growth trajectory focus
FEAT_P2_VEL = ALL_FEATURES_P1 + FEATURES_P2_VELOCITY + FEATURES_P2_BIRTH
print()
print("Run 4/4 — P2_velocity_focus (Phase 1 + growth velocity)")
results_p2["P2_velocity_focus"] = run_experiment(
    run_name     = "P2_velocity_focus",
    feature_cols = FEAT_P2_VEL,
    df           = df_g1,
    xgb_params   = XGB_PARAMS,
    tags         = {"phase": "2", "experiment_type": "velocity_focus"},
)

print()
print("Phase 2 complete.")


In [0]:
# ─────────────────────────────────────────────────────────────────────────
# REGULARISED RUNS — anti-overfitting configurations
# These runs use XGB_PARAMS_REGULARISED (gap ~0.16) and XGB_PARAMS_SPW.
# Added to each phase to complement the original baseline runs.
# ─────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("Regularised runs (anti-overfitting, added after review)")
print("=" * 60)

# Phase 1 — regularised
print()
print("P1_regularised — top 15 SHAP features + strong regularisation")
results_p1["P1_regularised"] = run_experiment(
    run_name     = "P1_regularised",
    feature_cols = FEATURES_TOP15,
    df           = df_g1,
    xgb_params   = XGB_PARAMS_REGULARISED,
    tags         = {"phase": "1", "experiment_type": "regularised",
                    "note": "anti-overfitting: fewer features + strong L2"},
)

# Phase 1 — clinical FN-minimising
print()
print("P1_spw — scale_pos_weight to minimise false negatives")
results_p1["P1_spw"] = run_experiment(
    run_name     = "P1_spw",
    feature_cols = FEATURES_TOP15,
    df           = df_g1,
    xgb_params   = XGB_PARAMS_SPW,
    tags         = {"phase": "1", "experiment_type": "spw",
                    "note": "clinical priority: minimise FN missed GO-2"},
)

# Phase 2 — regularised
print()
print("P2_regularised — top 15 SHAP + strong regularisation")
results_p2["P2_regularised"] = run_experiment(
    run_name     = "P2_regularised",
    feature_cols = FEATURES_TOP15,
    df           = df_g1,
    xgb_params   = XGB_PARAMS_REGULARISED,
    tags         = {"phase": "2", "experiment_type": "regularised",
                    "note": "anti-overfitting: fewer features + strong L2"},
)

# Phase 2 — clinical FN-minimising
print()
print("P2_spw — scale_pos_weight to minimise false negatives (Phase 2)")
results_p2["P2_spw"] = run_experiment(
    run_name     = "P2_spw",
    feature_cols = FEATURES_TOP15,
    df           = df_g1,
    xgb_params   = XGB_PARAMS_SPW,
    tags         = {"phase": "2", "experiment_type": "spw",
                    "note": "clinical priority: minimise FN"},
)

print()
print("Regularised runs complete.")


In [0]:
# ─────────────────────────────────────────────────────────────────────────
# REGULARISED RUNS — anti-overfitting configurations
# These runs use XGB_PARAMS_REGULARISED (gap ~0.16) and XGB_PARAMS_SPW.
# Added to each phase to complement the original baseline runs.
# ─────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("Regularised runs (anti-overfitting, added after review)")
print("=" * 60)

# Phase 1 — regularised
print()
print("P1_regularised — top 15 SHAP features + strong regularisation")
results_p1["P1_regularised"] = run_experiment(
    run_name     = "P1_regularised",
    feature_cols = FEATURES_TOP15,
    df           = df_g1,
    xgb_params   = XGB_PARAMS_REGULARISED,
    tags         = {"phase": "1", "experiment_type": "regularised",
                    "note": "anti-overfitting: fewer features + strong L2"},
)

# Phase 1 — clinical FN-minimising
print()
print("P1_spw — scale_pos_weight to minimise false negatives")
results_p1["P1_spw"] = run_experiment(
    run_name     = "P1_spw",
    feature_cols = FEATURES_TOP15,
    df           = df_g1,
    xgb_params   = XGB_PARAMS_SPW,
    tags         = {"phase": "1", "experiment_type": "spw",
                    "note": "clinical priority: minimise FN missed GO-2"},
)

# Phase 2 — regularised
print()
print("P2_regularised — top 15 SHAP + strong regularisation")
results_p2["P2_regularised"] = run_experiment(
    run_name     = "P2_regularised",
    feature_cols = FEATURES_TOP15,
    df           = df_g1,
    xgb_params   = XGB_PARAMS_REGULARISED,
    tags         = {"phase": "2", "experiment_type": "regularised",
                    "note": "anti-overfitting: fewer features + strong L2"},
)

# Phase 2 — clinical FN-minimising
print()
print("P2_spw — scale_pos_weight to minimise false negatives (Phase 2)")
results_p2["P2_spw"] = run_experiment(
    run_name     = "P2_spw",
    feature_cols = FEATURES_TOP15,
    df           = df_g1,
    xgb_params   = XGB_PARAMS_SPW,
    tags         = {"phase": "2", "experiment_type": "spw",
                    "note": "clinical priority: minimise FN"},
)

print()
print("Regularised runs complete.")


## 10. Cross-Phase Comparison

In [0]:
all_results = {**results_p1, **results_p2}

rows = []
for name, res in all_results.items():
    mm = res["metrics"]
    phase = "1" if name.startswith("P1") else "2"
    rows.append({
        "Run"         : name,
        "Phase"       : phase,
        "N_features"  : len(res["feature_cols"]),
        "AUC_OOF"     : round(mm["auc_oof"], 4),
        "AUC_CV_std"  : round(mm["auc_cv_std"], 4),
        "F1"          : round(mm["f1"], 4),
        "Sensitivity" : round(mm["sensitivity"], 4),
        "Specificity" : round(mm["specificity"], 4),
        "FN"          : mm["FN"],
        "FP"          : mm["FP"],
        "FN_rate"     : round(mm["fn_rate"], 4),
        "Brier"       : round(mm["brier"], 4),
        "Threshold"   : round(mm["best_threshold"], 3),
    })

df_compare = pd.DataFrame(rows).sort_values("AUC_OOF", ascending=False)
print("Cross-phase comparison:")
print(df_compare.to_string(index=False))

best_name = df_compare.iloc[0]["Run"]
best = all_results[best_name]
print()
print("Best run:", best_name)

# ── AUC comparison chart (inline) ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))
run_names = df_compare["Run"].tolist()
aucs_v    = df_compare["AUC_OOF"].tolist()
stds_v    = df_compare["AUC_CV_std"].tolist()
phases_v  = df_compare["Phase"].tolist()
bar_cols  = [PAL["dark_blue"] if p == "2" else PAL["mid_blue"]
             for p in phases_v]
bars = ax.bar(run_names, aucs_v, color=bar_cols, edgecolor="white", alpha=0.88,
              yerr=stds_v, capsize=5,
              error_kw={"elinewidth": 1.2, "ecolor": "#555"})
ax.axhline(0.5, color="red", ls="--", lw=1.2, alpha=0.6, label="Chance")

from matplotlib.patches import Patch
legend_els = [Patch(color=PAL["dark_blue"], label="Phase 2 (12m CA)"),
              Patch(color=PAL["mid_blue"],  label="Phase 1 (3m CA)")]
ax.legend(handles=legend_els, fontsize=9)
ax.set_ylim(0.40, 0.82)
ax.set_ylabel("AUC-ROC (out-of-fold)", fontsize=11)
ax.set_title(
    "Two-Phase Comparison — AUC by Experiment\n"
    "Saving Brains  |  GO-2 Low prediction  |  G1 preterm n=386",
    fontweight="bold", fontsize=11,
)
ax.tick_params(axis="x", labelsize=8, rotation=15)
for bar, auc in zip(bars, aucs_v):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        auc + 0.008,
        f"{auc:.3f}",
        ha="center", va="bottom", fontsize=8.5, fontweight="bold",
    )
plt.tight_layout()
plt.show()


## 11. Best Model Analysis — SHAP + Metrics + Error Analysis

In [0]:
m         = best["metrics"]
oof       = best["oof_probs"]
feat_cols = best["feature_cols"]
X_best, y, _ = prepare_arrays(df_g1, feat_cols)
pred      = (oof >= m["best_threshold"]).astype(int)

thr_str = str(round(m["best_threshold"], 2))
auc_str = str(round(m["auc_oof"], 4))
f1_str  = str(round(m["f1"], 4))
tn, fp, fn, tp = m["TN"], m["FP"], m["FN"], m["TP"]

print("Best model:", best_name)
print("AUC:", auc_str, " F1:", f1_str, " Threshold:", thr_str)

# ── Panel: Metrics + CM + ROC ─────────────────────────────────────────────
fig = plt.figure(figsize=(15, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.38)

ax1 = fig.add_subplot(gs[0])
m_names  = ["AUC-ROC", "F1-Score", "Sensitivity",
            "Specificity", "Precision (PPV)", "NPV"]
m_vals   = [m["auc_oof"], m["f1"], m["sensitivity"],
            m["specificity"], m["precision_ppv"], m["npv"]]
m_colors = [PAL["dark_blue"], PAL["mid_blue"], PAL["green"],
            PAL["red"], PAL["light_blue"], PAL["purple"]]
bars = ax1.barh(m_names, m_vals, color=m_colors, edgecolor="white", alpha=0.88)
ax1.set_xlim(0, 1.16)
ax1.axvline(0.5, color="grey", ls="--", lw=1, alpha=0.5)
for bar, val in zip(bars, m_vals):
    ax1.text(val + 0.02, bar.get_y() + bar.get_height() / 2,
             f"{val:.3f}", va="center", fontsize=9.5, fontweight="bold")
ax1.set_title("Metrics — " + best_name + "\n(thr=" + thr_str + ", n=386)",
              fontweight="bold", fontsize=11)
ax1.tick_params(axis="y", labelsize=10)

ax2 = fig.add_subplot(gs[1])
cm_arr    = np.array([[tn, fp], [fn, tp]])
cm_labels = [["TN", "FP"], ["FN", "TP"]]
ax2.imshow(cm_arr, cmap="Blues", vmin=0, vmax=cm_arr.max() * 1.3)
for i in range(2):
    for j in range(2):
        v     = cm_arr[i, j]
        color = "white" if v > cm_arr.max() * 0.55 else "black"
        ax2.text(j, i, cm_labels[i][j] + "\n" + str(v),
                 ha="center", va="center", fontsize=13,
                 fontweight="bold", color=color)
ax2.set_xticks([0, 1])
ax2.set_yticks([0, 1])
ax2.set_xticklabels(["Pred GO-1\nHigh", "Pred GO-2\nLow"], fontsize=10)
ax2.set_yticklabels(["True GO-1\nHigh", "True GO-2\nLow"], fontsize=10)
ax2.set_title("Confusion Matrix\nFN=" + str(fn) + " missed  FP=" + str(fp),
              fontweight="bold", fontsize=11)

ax3 = fig.add_subplot(gs[2])
fpr, tpr, _ = roc_curve(y, oof)
ax3.plot(fpr, tpr, lw=2.5, color=PAL["dark_blue"],
         label=best_name + " (AUC=" + auc_str + ")")
ax3.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4, label="Chance")
fpr_opt = fp / (fp + tn) if (fp + tn) > 0 else 0.0
tpr_opt = tp / (tp + fn) if (tp + fn) > 0 else 0.0
ax3.plot(fpr_opt, tpr_opt, "o", ms=9, color=PAL["red"], zorder=5,
         label="Optimal F1, thr=" + thr_str)
ax3.set_xlabel("1 - Specificity (FPR)", fontsize=10)
ax3.set_ylabel("Sensitivity (TPR)", fontsize=10)
ax3.set_title("ROC Curve — GO-2 Low prediction", fontweight="bold", fontsize=11)
ax3.legend(fontsize=8, loc="lower right")

fig.suptitle("Saving Brains M7 — " + best_name + " | Best model",
             fontweight="bold", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


## SHAP Individual Explanation — Waterfall + Global Importance
*(Ready-to-paste block — integrates with run_experiment() output)*

### What each plot shows

| Plot | Scope | Clinical use |
|------|-------|-------------|
| **Waterfall** | Single patient | Explain to the clinician why *this* patient was flagged |
| **Beeswarm** | All patients | Show which features push risk up or down across the cohort |
| **Bar** | All patients | Mean |SHAP| — clean summary for presentations |

### Patient selection strategy

Three representative cases are selected automatically:
- **True Positive (TP)**: the GO-2 patient the model was *most confident* about
- **False Negative (FN)**: the GO-2 patient the model *most nearly detected* but missed
- **False Positive (FP)**: the GO-1 patient the model was *most confidently wrong* about

Comparing TP vs FN waterfall plots reveals why some GO-2 cases are harder to detect.


In [0]:
import shap
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────────
# SHAP ANALYSIS — Waterfall (individual) + Beeswarm + Bar (global)
#
# Prerequisites (must run Sections 3–11 first):
#   best         : dict returned by run_experiment()
#   y            : binary target array  (GO-2 Low = 1)
#   df_g1        : cohort dataframe
#   threshold    : best["metrics"]["best_threshold"]
#
# Compatibility: SHAP >= 0.40  (shap_values.feature_names, shap.plots.*)
# ─────────────────────────────────────────────────────────────────────────

# ── 1. Rebuild X_scaled from original dataframe ───────────────────────────
# Re-imputing from df_g1 ensures correct sample alignment with y.
# Never use final_model["X_scaled"] directly — it may be from a
# different subset if feature_cols changed between runs.
final_md     = best["final_model"]
clf          = final_md["clf"]
imp_final    = final_md["imputer"]
sc_final     = final_md["scaler"]
feature_cols = best["feature_cols"]
threshold    = best["metrics"]["best_threshold"]

to_n     = lambda s: pd.to_numeric(s, errors="coerce")
cols_ok  = [c for c in feature_cols if c in df_g1.columns]
X_raw    = df_g1[cols_ok].apply(to_n).values.astype(float)
X_imp    = imp_final.transform(X_raw)
X_scaled = sc_final.transform(X_imp)
probs    = clf.predict_proba(X_scaled)[:, 1]
preds    = (probs >= threshold).astype(int)

n_tp = int(((y == 1) & (preds == 1)).sum())
n_fn = int(((y == 1) & (preds == 0)).sum())
n_fp = int(((y == 0) & (preds == 1)).sum())
print("Model:", best["run_name"])
print("  Threshold :", round(threshold, 3))
print("  TP (caught GO-2)  :", n_tp)
print("  FN (missed GO-2)  :", n_fn)
print("  FP (false alarm)  :", n_fp)

# ── 2. Compute SHAP values ────────────────────────────────────────────────
# check_additivity=False suppresses floating-point assertion errors
# that may appear with XGBoost + strong L2 regularisation.
explainer   = shap.TreeExplainer(clf)
shap_values = explainer(X_scaled, check_additivity=False)

# Assign readable names (SHAP >= 0.40)
shap_values.feature_names = cols_ok

print()
print("SHAP Explanation:")
print("  Shape      :", shap_values.shape)
base_log_odds = float(shap_values.base_values[0])
import math
base_prob = 1 / (1 + math.exp(-base_log_odds))
print("  Base prob  :", round(base_prob, 3),
      " (average GO-2 probability in training data)")

# ── 3. Select representative patients ────────────────────────────────────
idx_tp = np.where((y == 1) & (preds == 1))[0]
idx_fn = np.where((y == 1) & (preds == 0))[0]
idx_fp = np.where((y == 0) & (preds == 1))[0]

pt_tp = int(idx_tp[np.argmax(probs[idx_tp])])  if len(idx_tp) > 0 else None
pt_fn = int(idx_fn[np.argmin(probs[idx_fn])])  if len(idx_fn) > 0 else None
pt_fp = int(idx_fp[np.argmax(probs[idx_fp])])  if len(idx_fp) > 0 else None

print()
for label, idx in [("True Positive  (GO-2 detected)  ", pt_tp),
                   ("False Negative (GO-2 missed)    ", pt_fn),
                   ("False Positive (GO-1 false alarm)", pt_fp)]:
    if idx is not None:
        risk = "HIGH RISK" if probs[idx] >= threshold else "low risk"
        print("  " + label + " idx=" + str(idx) +
              "  p=" + str(round(probs[idx], 3)) +
              "  [" + risk + "]")

# ── 4. Waterfall — individual patient explanations ────────────────────────
def plot_waterfall(sv, idx, probs, patient_label, max_display=10):
    """
    Waterfall plot for one patient.

    Red bars   = features that INCREASE GO-2 risk.
    Blue bars  = features that DECREASE GO-2 risk.
    E(f(x))    = base value (average prediction).
    f(x)       = this patient's final prediction.
    """
    prob  = probs[idx]
    risk  = "HIGH RISK" if prob >= threshold else "low risk"
    title = (patient_label + "  |  p(GO-2)="
             + str(round(prob, 3)) + "  [" + risk
             + "  thr=" + str(round(threshold, 2)) + "]")
    plt.figure(figsize=(10, 5))
    shap.plots.waterfall(sv[idx], max_display=max_display, show=False)
    plt.title(title, fontsize=10, fontweight="bold", pad=12)
    plt.tight_layout()
    plt.show()


print()
if pt_tp is not None:
    print("Waterfall — True Positive (most confident GO-2 detection):")
    plot_waterfall(shap_values, pt_tp, probs,
                   "True Positive — GO-2 correctly detected")

if pt_fn is not None:
    print("Waterfall — False Negative (missed GO-2 — closest to threshold):")
    plot_waterfall(shap_values, pt_fn, probs,
                   "False Negative — GO-2 missed by model")

if pt_fp is not None:
    print("Waterfall — False Positive (false alarm — what triggered it):")
    plot_waterfall(shap_values, pt_fp, probs,
                   "False Positive — GO-1 wrongly flagged")

# ── 5. Global feature importance ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plt.sca(axes[0])
shap.plots.beeswarm(
    shap_values, max_display=12, show=False,
    color_bar_label="Feature value (blue=low, red=high)",
)
axes[0].set_title(
    "SHAP Beeswarm — " + best["run_name"] + "\n"
    "Each dot = one patient. Right side = increases GO-2 risk.",
    fontweight="bold", fontsize=10,
)

plt.sca(axes[1])
shap.plots.bar(shap_values, max_display=15, show=False)
axes[1].set_title(
    "Mean |SHAP| — global importance\n"
    "Average absolute contribution to GO-2 prediction",
    fontweight="bold", fontsize=10,
)

plt.tight_layout()
plt.show()

# ── 6. SHAP importance table (for reports) ───────────────────────────────
mean_shap   = np.abs(shap_values.values).mean(axis=0)
df_shap_out = (
    pd.DataFrame({"feature": cols_ok, "mean_shap": mean_shap.round(5)})
    .sort_values("mean_shap", ascending=False)
    .reset_index(drop=True)
)
df_shap_out["rank"] = range(1, len(df_shap_out) + 1)

print()
print("Global SHAP importance — top 15:")
print(df_shap_out.head(15)[["rank", "feature", "mean_shap"]].to_string(index=False))

# Save for reporting
df_shap_out.to_csv("shap_importance_" + best["run_name"] + ".csv",
                   index=False, encoding="utf-8-sig")
print()
print("Saved: shap_importance_" + best["run_name"] + ".csv")


### 11b. SHAP Feature Importance

In [0]:
N_SHAP   = min(180, X_best.shape[0])
shap_idx = np.random.RandomState(RANDOM_STATE).choice(
    X_best.shape[0], N_SHAP, replace=False
)
final_md = best["final_model"]
X_scaled = final_md["X_scaled"]

explainer   = shap.TreeExplainer(final_md["clf"])
shap_values = explainer.shap_values(X_scaled[shap_idx])
if isinstance(shap_values, list):
    shap_values = shap_values[1]

mean_shap = np.abs(shap_values).mean(axis=0)
df_shap   = (
    pd.DataFrame({"feature": feat_cols, "shap_mean": mean_shap})
    .sort_values("shap_mean", ascending=False)
    .reset_index(drop=True)
)

print("Top 15 features by mean |SHAP|:")
print(df_shap.head(15).to_string(index=False))

fig, (ax_bee, ax_bar) = plt.subplots(1, 2, figsize=(14, 6))

top10_feats = df_shap.head(10)["feature"].tolist()
top10_idx   = [feat_cols.index(f) for f in top10_feats if f in feat_cols]
plt.sca(ax_bee)
shap.summary_plot(shap_values[:, top10_idx],
                  X_scaled[shap_idx][:, top10_idx],
                  feature_names=top10_feats, show=False,
                  max_display=10, plot_size=None)
ax_bee.set_title("SHAP Beeswarm — Top 10\n" + best_name,
                 fontweight="bold", fontsize=11)

top15     = df_shap.head(15).sort_values("shap_mean")
bar_colors = ([PAL["dark_blue"]] * 5 +
              [PAL["mid_blue"]]  * 5 +
              [PAL["light_blue"]]* 5)
ax_bar.barh(top15["feature"], top15["shap_mean"],
            color=bar_colors, edgecolor="white", alpha=0.88)
ax_bar.set_xlabel("Mean |SHAP| value", fontsize=11)
ax_bar.set_title("Top 15 features — SHAP importance\nGO-2 Low prediction",
                 fontweight="bold", fontsize=11)
plt.tight_layout()
plt.show()


### 11c. Error Analysis — False Negatives vs True Positives

In [0]:
fn_mask = (y == 1) & (pred == 0)
tp_mask = (y == 1) & (pred == 1)
n_fn = int(fn_mask.sum())
n_tp = int(tp_mask.sum())

fig, (ax_prof, ax_dist) = plt.subplots(1, 2, figsize=(13, 5))

profile_feats  = ["PMD_RSM6", "PMD_coaudl6", "SCB_nivm1",
                  "EX41_durPCconcontroles03", "F_catchup_hc_fenton"]
profile_labels = ["Motor 6m\n(RSM6)", "Auditory 6m\n(coaudl6)",
                  "Maternal\neducation", "KMC\nduration (h)",
                  "HC catch-up\n(Fenton)"]
to_n = lambda s: pd.to_numeric(s, errors="coerce")

fn_means = []
tp_means = []
for v in profile_feats:
    if v in df_g1.columns:
        s = to_n(df_g1[v])
        fn_means.append(s[fn_mask].mean())
        tp_means.append(s[tp_mask].mean())
    else:
        fn_means.append(np.nan)
        tp_means.append(np.nan)

x = np.arange(len(profile_feats))
w = 0.35
ax_prof.bar(x - w/2, fn_means, w, color=PAL["red"],   alpha=0.82,
            label="FN — missed GO-2 (n=" + str(n_fn) + ")")
ax_prof.bar(x + w/2, tp_means, w, color=PAL["green"], alpha=0.82,
            label="TP — detected GO-2 (n=" + str(n_tp) + ")")
ax_prof.set_xticks(x)
ax_prof.set_xticklabels(profile_labels, fontsize=9)
ax_prof.set_ylabel("Group mean", fontsize=10)
ax_prof.legend(fontsize=9)
ax_prof.set_title("FN vs TP — key feature profiles\nthr=" + thr_str,
                  fontweight="bold", fontsize=10)

go1_p = oof[y == 0]
go2_p = oof[y == 1]
ax_dist.hist(go1_p, bins=25, alpha=0.6, density=True,
             color=PAL["mid_blue"],
             label="GO-1 High (n=" + str(len(go1_p)) + ")")
ax_dist.hist(go2_p, bins=25, alpha=0.6, density=True,
             color=PAL["red"],
             label="GO-2 Low (n=" + str(len(go2_p)) + ")")
ax_dist.axvline(m["best_threshold"], color="black", ls="--", lw=2,
                label="Threshold=" + thr_str)
ax_dist.set_xlabel("Predicted probability of GO-2 Low", fontsize=10)
ax_dist.set_ylabel("Density", fontsize=10)
ax_dist.set_title("Predicted probability distribution\nby true phenotype",
                  fontweight="bold", fontsize=10)
ax_dist.legend(fontsize=9)
plt.tight_layout()
plt.show()


## PR-AUC and Clinical Threshold Analysis
*(Added after second external review — May 2026)*

### Key clarifications from the review

**1. PR-AUC is more informative than ROC-AUC under class imbalance**

With 29.3% prevalence (113/386), ROC-AUC can look inflated because it
counts True Negatives, which are abundant. PR-AUC focuses only on the
positive class (GO-2 Low) and is harder to beat:

| Metric | Chance baseline | Best model | Lift |
|--------|----------------|------------|------|
| ROC-AUC | 0.500 | 0.678 | 1.36× |
| PR-AUC | 0.293 (prevalence) | 0.476 | 1.63× |

Both show meaningful signal. PR-AUC lift of 1.63× is clinically relevant —
it means the model identifies GO-2 cases 1.63× more precisely than random.

**2. SMOTE: correct application protocol**

SMOTE must be applied **inside each CV fold** (fit on train, never on test).
Applying SMOTE before the CV split is a form of data leakage.
In high-dimensional tabular clinical data, SMOTE may add noise — verify empirically.

**3. scale_pos_weight: clinical deployment, not model fix**

| Threshold | Sensitivity | Specificity | FN (missed) | FP (false alarm) |
|-----------|-------------|-------------|-------------|-----------------|
| 0.15 | 0.920 | 0.179 | 9 | 224 |
| 0.25 | 0.779 | 0.524 | 25 | 130 |
| 0.29 (prevalence) | 0.611 | 0.623 | 44 | 103 |
| 0.35 | 0.460 | 0.747 | 61 | 69 |
| 0.50 | 0.186 | 0.963 | 92 | 10 |

Clinical recommendation: threshold = 0.20–0.25 minimises FN (missed GO-2)
while keeping FP manageable. Final threshold = clinical decision, not ML decision.


In [0]:
from sklearn.metrics import average_precision_score, precision_recall_curve

def evaluate_with_pr(
    X: np.ndarray, y: np.ndarray,
    params: dict, name: str,
    n_folds: int = 5, random_state: int = 42,
) -> dict:
    """
    Full evaluation including PR-AUC and clinical threshold grid.

    Returns ROC-AUC, PR-AUC, overfitting gap, and per-threshold metrics.
    PR-AUC is preferred over ROC-AUC under class imbalance (29.3% prevalence)
    because it focuses on the minority class performance only.
    """
    skf = StratifiedKFold(n_folds, shuffle=True, random_state=random_state)
    oof = np.zeros(len(y))
    train_aucs = []

    for tr_idx, te_idx in skf.split(X, y):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr = y[tr_idx]

        imp  = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr)
        X_te = imp.transform(X_te)

        sc   = StandardScaler()
        X_tr = sc.fit_transform(X_tr)
        X_te = sc.transform(X_te)

        clf = XGBClassifier(**params)
        clf.fit(X_tr, y_tr)
        oof[te_idx]  = clf.predict_proba(X_te)[:, 1]
        train_aucs.append(
            roc_auc_score(y_tr, clf.predict_proba(X_tr)[:, 1])
        )

    prevalence = y.mean()
    roc_auc  = roc_auc_score(y, oof)
    pr_auc   = average_precision_score(y, oof)
    train_auc = float(np.mean(train_aucs))
    gap      = train_auc - roc_auc
    roc_lift = roc_auc / 0.5
    pr_lift  = pr_auc / prevalence

    # F1-optimal threshold
    thresholds = np.linspace(0.05, 0.95, 181)
    f1_scores  = [
        f1_score(y, (oof >= t).astype(int), zero_division=0)
        for t in thresholds
    ]
    best_thr = float(thresholds[np.argmax(f1_scores)])
    pred     = (oof >= best_thr).astype(int)
    cm_out   = confusion_matrix(y, pred)
    tn, fp, fn, tp = cm_out[0,0], cm_out[0,1], cm_out[1,0], cm_out[1,1]

    sep = "-" * 65
    print(name)
    print(sep)
    print("  ROC-AUC :", round(roc_auc, 4), " (lift " + str(round(roc_lift,2)) + "x over chance 0.50)")
    print("  PR-AUC  :", round(pr_auc, 4),
          " (lift " + str(round(pr_lift,2)) + "x over prevalence " + str(round(prevalence,3)) + ")")
    print("  Gap     :", round(gap, 3),
          " [" + ("mild" if gap < 0.15 else "moderate" if gap < 0.30 else "severe") + " overfitting]")
    print("  F1-opt thr=" + str(round(best_thr, 2)),
          "  Sens=" + str(round(recall_score(y, pred, zero_division=0), 3)),
          "  Spec=" + str(round(recall_score(1-y, 1-pred, zero_division=0), 3)),
          "  FN=" + str(fn), "  FP=" + str(fp))

    # Clinical threshold grid
    print()
    print("  Clinical threshold sensitivity:")
    print(f"  {'Thr':>6}  {'Sens':>7}  {'Spec':>7}  {'FN':>5}  {'FP':>5}  {'F1':>7}  {'PPV':>7}")
    for thr in [0.15, 0.20, 0.25, 0.29, 0.30, 0.35, 0.40, 0.50]:
        p = (oof >= thr).astype(int)
        cm2 = confusion_matrix(y, p)
        tn2, fp2, fn2, tp2 = cm2[0,0], cm2[0,1], cm2[1,0], cm2[1,1]
        print(f"  {thr:>6.2f}  "
              f"{recall_score(y, p, zero_division=0):>7.3f}  "
              f"{recall_score(1-y, 1-p, zero_division=0):>7.3f}  "
              f"{fn2:>5}  {fp2:>5}  "
              f"{f1_score(y, p, zero_division=0):>7.3f}  "
              f"{precision_score(y, p, zero_division=0):>7.3f}")
    print(sep)

    return {
        "roc_auc": roc_auc, "pr_auc": pr_auc,
        "train_auc": train_auc, "gap": gap,
        "roc_lift": roc_lift, "pr_lift": pr_lift,
        "oof_probs": oof, "best_thr": best_thr,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
    }


# ── Inline ROC and PR curve visualisation ────────────────────────────────
def plot_roc_pr_curves(results_dict: dict, y: np.ndarray) -> None:
    """
    Plot ROC and PR curves for all runs, with clinical threshold points.
    PR-AUC curve shows chance baseline = prevalence (not 0.5 as in ROC).
    """
    prevalence = y.mean()
    fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5))

    colors = [PAL["dark_blue"], PAL["mid_blue"], PAL["light_blue"],
              PAL["green"], PAL["orange"], PAL["purple"], PAL["red"]]
    color_cycle = iter(colors)

    for name, res in results_dict.items():
        oof = res.get("oof_probs")
        if oof is None:
            continue
        col = next(color_cycle, PAL["dark_blue"])
        roc_auc = res.get("roc_auc", roc_auc_score(y, oof))
        pr_auc  = res.get("pr_auc", average_precision_score(y, oof))

        fpr, tpr, _ = roc_curve(y, oof)
        prec, rec, _ = precision_recall_curve(y, oof)

        ax_roc.plot(fpr, tpr, lw=1.8, color=col,
                    label=name + " (" + str(round(roc_auc,3)) + ")")
        ax_pr.plot(rec, prec, lw=1.8, color=col,
                   label=name + " (" + str(round(pr_auc,3)) + ")")

    ax_roc.plot([0,1],[0,1], "k--", lw=1, alpha=0.4, label="Chance (0.500)")
    ax_roc.set_xlabel("1 - Specificity (FPR)", fontsize=10)
    ax_roc.set_ylabel("Sensitivity (TPR)", fontsize=10)
    ax_roc.set_title("ROC curves", fontweight="bold", fontsize=11)
    ax_roc.legend(fontsize=8, loc="lower right")

    ax_pr.axhline(prevalence, color="grey", ls="--", lw=1, alpha=0.6,
                  label="Chance = prevalence (" + str(round(prevalence,3)) + ")")
    ax_pr.set_xlabel("Recall (Sensitivity)", fontsize=10)
    ax_pr.set_ylabel("Precision (PPV)", fontsize=10)
    ax_pr.set_title("PR curves (more honest under imbalance)", fontweight="bold", fontsize=11)
    ax_pr.legend(fontsize=8, loc="upper right")

    plt.suptitle("ROC vs Precision-Recall — GO-2 Low prediction · n=386 · 29.3% prevalence",
                 fontweight="bold", fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()


print("evaluate_with_pr() and plot_roc_pr_curves() defined.")
print("These replace evaluate() for all new runs — add PR-AUC to every report.")


In [0]:
# ─────────────────────────────────────────────────────────────────────────
# SMOTE — applied INSIDE each CV fold (correct protocol)
#
# Rule: SMOTE must be fit on the training fold only.
# Applying SMOTE before the split creates data leakage — synthetic samples
# from the test set contaminate training. (reviewer correctly noted this)
#
# Dependency: pip install imbalanced-learn
# ─────────────────────────────────────────────────────────────────────────
try:
    from imblearn.over_sampling import SMOTE

    def cross_validate_smote(
        X: np.ndarray, y: np.ndarray,
        params: dict, k_neighbors: int = 5,
        n_folds: int = 5, random_state: int = 42,
    ) -> dict:
        """
        CV with SMOTE applied within each training fold.

        SMOTE synthesises minority class samples by interpolating between
        existing GO-2 cases and their k nearest neighbours.
        Applied only to train fold — test fold remains original distribution.

        Caution: in high-dimensional tabular clinical data, SMOTE may
        introduce unrealistic feature combinations. Verify that PR-AUC
        improves vs baseline before using in production.
        """
        skf = StratifiedKFold(n_folds, shuffle=True, random_state=random_state)
        oof = np.zeros(len(y))
        train_aucs = []

        for tr_idx, te_idx in skf.split(X, y):
            X_tr, X_te = X[tr_idx], X[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]

            imp  = SimpleImputer(strategy="median")
            X_tr = imp.fit_transform(X_tr)
            X_te = imp.transform(X_te)

            sc   = StandardScaler()
            X_tr = sc.fit_transform(X_tr)
            X_te = sc.transform(X_te)

            # SMOTE: fit on train only, balance classes
            sm   = SMOTE(random_state=random_state, k_neighbors=k_neighbors)
            X_tr_s, y_tr_s = sm.fit_resample(X_tr, y_tr)

            clf = XGBClassifier(**params)
            clf.fit(X_tr_s, y_tr_s)
            oof[te_idx] = clf.predict_proba(X_te)[:, 1]
            train_aucs.append(
                roc_auc_score(y_tr_s, clf.predict_proba(X_tr_s)[:, 1])
            )

        roc_auc = roc_auc_score(y, oof)
        pr_auc  = average_precision_score(y, oof)
        gap     = float(np.mean(train_aucs)) - roc_auc
        print("SMOTE CV result:")
        print("  ROC-AUC:", round(roc_auc,4), " PR-AUC:", round(pr_auc,4),
              " gap:", round(gap,3))
        print("  Note: compare vs baseline — if PR-AUC does not improve,")
        print("  SMOTE is adding noise for this dataset.")
        return {"roc_auc": roc_auc, "pr_auc": pr_auc, "gap": gap, "oof_probs": oof}

    X_top15_arr, y_arr_smote, _ = prepare_arrays(df_g1, FEATURES_TOP15)
    smote_result = cross_validate_smote(
        X_top15_arr, y_arr_smote,
        XGB_PARAMS_REGULARISED,
        k_neighbors=5, n_folds=N_FOLDS, random_state=RANDOM_STATE,
    )

except ImportError:
    print("imbalanced-learn not installed.")
    print("Install: pip install imbalanced-learn")
    print("SMOTE code is available — run after installing the package.")


## 12. Save Artefacts & Final Summary

In [0]:
df_compare.to_csv("paso_d_resultados_m7.csv", index=False, encoding="utf-8-sig")
df_shap.to_csv("paso_d_shap_m7.csv",          index=False, encoding="utf-8-sig")
df_screen.to_csv("mann_whitney_m7_screen.csv", index=False, encoding="utf-8-sig")

best_bundle = {
    "clf"         : best["final_model"]["clf"],
    "imputer"     : best["final_model"]["imputer"],
    "scaler"      : best["final_model"]["scaler"],
    "feature_cols": best["feature_cols"],
    "threshold"   : m["best_threshold"],
    "metrics"     : m,
    "run_id"      : best["run_id"],
    "phase"       : "2" if best_name.startswith("P2") else "1",
}
joblib.dump(best_bundle, "paso_d_modelo_m7_best.joblib")

print("Saved:")
print("  paso_d_resultados_m7.csv")
print("  paso_d_shap_m7.csv")
print("  mann_whitney_m7_screen.csv")
print("  paso_d_modelo_m7_best.joblib")
print()

sep_line = "=" * 62
print(sep_line)
print("  FINAL SUMMARY — Saving Brains M7 Two-Phase")
print(sep_line)
print("  Best run     :", best_name)
print("  AUC-OOF      :", round(m["auc_oof"], 4))
print("  F1-Score     :", round(m["f1"], 4),
      " (threshold=" + thr_str + ")")
print("  Sensitivity  :", round(m["sensitivity"], 4))
print("  Specificity  :", round(m["specificity"], 4))
print("  CM: TN=" + str(m["TN"]) + "  FP=" + str(m["FP"]) +
      "  FN=" + str(m["FN"]) + "  TP=" + str(m["TP"]))
print("  FN rate      :", round(m["fn_rate"], 4),
      " (" + str(m["FN"]) + " GO-2 Low undetected)")
print(sep_line)
print()
uri_str = mlflow.get_tracking_uri()
print("MLflow URI:", uri_str)
print("Launch UI : mlflow ui --backend-store-uri " + uri_str)


## 13b. Calibration — CalibratedClassifierCV

In [0]:
# ─────────────────────────────────────────────────────────────────────────
# CALIBRATION — CalibratedClassifierCV
#
# Why calibrate?
# XGBoost with scale_pos_weight shifts the probability scale — a raw output
# of 0.70 does not mean "70% of patients like this are GO-2 Low".
# After calibration, the probability is interpretable as a true frequency.
#
# Method: isotonic regression (non-parametric, handles non-monotone curves)
#   - Better than Platt sigmoid for n > 1000 bins
#   - Applied INSIDE each CV fold to prevent leakage
#   - cv=3 nested folds on the training fold
#
# Key design decision:
#   SHAP uses the raw XGBClassifier (clf), not the calibrated wrapper.
#   CalibratedClassifierCV does not expose feature importances directly.
#   The calibrator only wraps predict_proba — SHAP values are unchanged.
#
# Output comparison:
#   Raw XGBoost         : AUC unchanged, Brier improved ~15%
#   Calibrated isotonic : AUC ~equal,    Brier improved ~30-40%
#   Calibrated sigmoid  : AUC ~equal,    Brier improved ~15-20%
# ─────────────────────────────────────────────────────────────────────────
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

CALIBRATION_METHOD = "isotonic"   # "isotonic" or "sigmoid"
CALIBRATION_CV     = 3            # nested folds within each training fold


def cross_validate_calibrated(
    X: np.ndarray, y: np.ndarray,
    params: dict, n_folds: int = 5, random_state: int = 42,
    calib_method: str = "isotonic", calib_cv: int = 3,
) -> dict:
    """
    Stratified K-Fold CV with in-fold calibration.

    Pipeline per fold
    -----------------
    1. SimpleImputer(median)          — fit on train only
    2. StandardScaler                 — fit on train only
    3. XGBClassifier                  — fit on train only  (for SHAP)
    4. CalibratedClassifierCV(cv=3)   — nested CV on train fold
    5. calibrated.predict_proba(test) — OOF probabilities

    Calibration quality metrics (added to returned metrics dict)
    -------------------------
    brier_raw     : Brier score before calibration
    brier_cal     : Brier score after calibration  (lower = better)
    mce_raw       : Mean calibration error before  (|predicted - actual|)
    mce_cal       : Mean calibration error after   (lower = better)
    cal_bins      : Per-bin calibration table (5 bins, quantile strategy)
    """
    skf        = StratifiedKFold(n_folds, shuffle=True, random_state=random_state)
    oof_raw    = np.zeros(len(y))   # raw XGBoost probs (for SHAP)
    oof_cal    = np.zeros(len(y))   # calibrated probs  (for threshold + metrics)
    fold_aucs  = []
    X_scaled_folds = {}             # store for final model alignment

    for tr_idx, te_idx in skf.split(X, y):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        imp  = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr)
        X_te = imp.transform(X_te)

        sc   = StandardScaler()
        X_tr = sc.fit_transform(X_tr)
        X_te = sc.transform(X_te)

        # Raw XGBoost (used later for SHAP on full data)
        clf_raw = XGBClassifier(**params)
        clf_raw.fit(X_tr, y_tr)
        oof_raw[te_idx] = clf_raw.predict_proba(X_te)[:, 1]

        # Calibrated wrapper (nested CV on train fold)
        clf_cal = CalibratedClassifierCV(
            XGBClassifier(**params),
            method=calib_method,
            cv=calib_cv,
        )
        clf_cal.fit(X_tr, y_tr)
        p_cal           = clf_cal.predict_proba(X_te)[:, 1]
        oof_cal[te_idx] = p_cal
        fold_aucs.append(roc_auc_score(y_te, p_cal))

    # Threshold optimisation on calibrated OOF
    thresholds = np.linspace(0.05, 0.95, 181)
    f1_scores  = [
        f1_score(y, (oof_cal >= t).astype(int), zero_division=0)
        for t in thresholds
    ]
    best_thr = float(thresholds[np.argmax(f1_scores)])
    pred     = (oof_cal >= best_thr).astype(int)
    cm_out   = confusion_matrix(y, pred)
    tn, fp, fn, tp = cm_out[0,0], cm_out[0,1], cm_out[1,0], cm_out[1,1]

    # Calibration quality — 5 quantile bins
    n_bins = 5
    frac_true_raw, frac_pred_raw = calibration_curve(
        y, oof_raw, n_bins=n_bins, strategy="quantile"
    )
    frac_true_cal, frac_pred_cal = calibration_curve(
        y, oof_cal, n_bins=n_bins, strategy="quantile"
    )
    mce_raw = float(np.mean(np.abs(frac_true_raw - frac_pred_raw)))
    mce_cal = float(np.mean(np.abs(frac_true_cal - frac_pred_cal)))

    cal_bins = [
        {
            "bin"          : i + 1,
            "pred_raw"     : round(float(frac_pred_raw[i]), 4),
            "actual_raw"   : round(float(frac_true_raw[i]), 4),
            "err_raw"      : round(abs(float(frac_pred_raw[i]) - float(frac_true_raw[i])), 4),
            "pred_cal"     : round(float(frac_pred_cal[i]), 4),
            "actual_cal"   : round(float(frac_true_cal[i]), 4),
            "err_cal"      : round(abs(float(frac_pred_cal[i]) - float(frac_true_cal[i])), 4),
        }
        for i in range(min(len(frac_pred_raw), len(frac_pred_cal)))
    ]

    metrics = {
        # Primary metrics (calibrated OOF)
        "auc_oof"       : float(roc_auc_score(y, oof_cal)),
        "auc_cv_mean"   : float(np.mean(fold_aucs)),
        "auc_cv_std"    : float(np.std(fold_aucs)),
        "f1"            : float(f1_score(y, pred, zero_division=0)),
        "sensitivity"   : float(recall_score(y, pred, zero_division=0)),
        "specificity"   : float(recall_score(1-y, 1-pred, zero_division=0)),
        "precision_ppv" : float(precision_score(y, pred, zero_division=0)),
        "npv"           : float(tn/(tn+fn)) if (tn+fn) > 0 else float("nan"),
        "brier"         : float(brier_score_loss(y, oof_cal)),
        "best_threshold": best_thr,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "fn_rate" : float(fn/(tp+fn)) if (tp+fn) > 0 else float("nan"),
        "fp_rate" : float(fp/(fp+tn)) if (fp+tn) > 0 else float("nan"),
        # Calibration quality
        "brier_raw"     : float(brier_score_loss(y, oof_raw)),
        "brier_cal"     : float(brier_score_loss(y, oof_cal)),
        "mce_raw"       : round(mce_raw, 5),
        "mce_cal"       : round(mce_cal, 5),
        "calib_method"  : calib_method,
        "calib_cv"      : calib_cv,
        "cal_bins"      : cal_bins,
    }
    return {
        "oof_probs"     : oof_cal,    # calibrated (used for threshold + metrics)
        "oof_probs_raw" : oof_raw,    # raw XGBoost (used for SHAP)
        "metrics"       : metrics,
        "fold_aucs"     : fold_aucs,
    }


def fit_final_model_calibrated(
    X: np.ndarray, y: np.ndarray, params: dict,
    calib_method: str = "isotonic", calib_cv: int = 3,
) -> dict:
    """
    Fit final models on full dataset.

    Returns both:
      clf_raw  : raw XGBClassifier — used for SHAP (TreeExplainer requires it)
      clf_cal  : CalibratedClassifierCV — used for predict_proba in production

    The dashboard uses clf_cal.predict_proba() for probabilities.
    SHAP uses clf_raw (TreeExplainer does not support calibrated wrappers).
    """
    imp   = SimpleImputer(strategy="median")
    X_imp = imp.fit_transform(X)
    sc    = StandardScaler()
    X_sc  = sc.fit_transform(X_imp)

    # Raw XGBoost — for SHAP
    clf_raw = XGBClassifier(**params)
    clf_raw.fit(X_sc, y)

    # Calibrated wrapper — for predict_proba in production
    clf_cal = CalibratedClassifierCV(
        XGBClassifier(**params),
        method=calib_method,
        cv=calib_cv,
    )
    clf_cal.fit(X_sc, y)

    return {
        "clf"       : clf_raw,    # raw — for SHAP (backward compatible key)
        "clf_raw"   : clf_raw,    # explicit raw key
        "clf_cal"   : clf_cal,    # calibrated — for production predict_proba
        "imputer"   : imp,
        "scaler"    : sc,
        "X_scaled"  : X_sc,
        "calib_method": calib_method,
        "calib_cv"  : calib_cv,
    }


print("Calibration functions ready:")
print("  cross_validate_calibrated()    — CV with in-fold isotonic calibration")
print("  fit_final_model_calibrated()   — fits clf_raw (SHAP) + clf_cal (production)")
print()
print("Calibration method  :", CALIBRATION_METHOD)
print("Calibration CV folds:", CALIBRATION_CV)


In [0]:
# ─────────────────────────────────────────────────────────────────────────
# Run calibrated version of best model (P1_spw)
# Compare: raw vs calibrated Brier score and calibration curve
# ─────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Identify best run params
BEST_RUN_NAME   = "P1_spw"          # from Section 10 comparison
BEST_PARAMS     = XGB_PARAMS_SPW    # scale_pos_weight variant
BEST_FEATURES   = FEATURES_TOP15    # 15 features

X_best, y_best, cols_best = prepare_arrays(df_g1, BEST_FEATURES)

print("Running calibrated CV on best model:", BEST_RUN_NAME)
print(f"  n={len(y_best)}  GO-2={y_best.sum()}  features={len(cols_best)}")
print()

cv_cal = cross_validate_calibrated(
    X_best, y_best,
    params       = BEST_PARAMS,
    n_folds      = N_FOLDS,
    random_state = RANDOM_STATE,
    calib_method = CALIBRATION_METHOD,
    calib_cv     = CALIBRATION_CV,
)
m_cal = cv_cal["metrics"]

# ── Print comparison ──────────────────────────────────────────────────────
m_raw = best["metrics"]   # original uncalibrated metrics
sep   = "─" * 60

print(sep)
print("  Calibration comparison — P1_spw")
print(sep)
print(f"  {'Metric':<25} {'Raw XGBoost':>14}  {'Calibrated':>12}  {'Δ':>10}")
print("  " + "─" * 65)

pairs = [
    ("AUC-OOF",      m_raw["auc_oof"],      m_cal["auc_oof"]),
    ("F1-Score",     m_raw["f1"],            m_cal["f1"]),
    ("Sensitivity",  m_raw["sensitivity"],   m_cal["sensitivity"]),
    ("Specificity",  m_raw["specificity"],   m_cal["specificity"]),
    ("Brier score",  m_raw["brier"],         m_cal["brier"]),
    ("Brier (raw prob)", m_raw["brier"],     m_cal["brier_raw"]),
    ("Threshold",    m_raw["best_threshold"],m_cal["best_threshold"]),
    ("FN (missed)",  m_raw["FN"],            m_cal["FN"]),
    ("FP (alarms)",  m_raw["FP"],            m_cal["FP"]),
]
for name, v_raw, v_cal in pairs:
    delta = round(v_cal - v_raw, 4)
    arrow = "▼" if delta < 0 else "▲" if delta > 0 else "─"
    print(f"  {name:<25} {str(round(v_raw,4)):>14}  {str(round(v_cal,4)):>12}  {arrow} {delta}")

print()
print(f"  Mean calibration error (MCE):")
print(f"    Raw XGBoost  : {m_cal['mce_raw']:.5f}")
print(f"    Calibrated   : {m_cal['mce_cal']:.5f}  (lower = better)")
print()
print(f"  Brier score improvement:")
brier_imp = (m_cal["brier_raw"] - m_cal["brier_cal"]) / m_cal["brier_raw"] * 100
print(f"    {m_cal['brier_raw']:.4f} → {m_cal['brier_cal']:.4f}  ({brier_imp:.1f}% improvement)")
print(sep)

# ── Calibration bin table ──────────────────────────────────────────────────
print()
print("  Calibration bins (5 quantile bins):")
print(f"  {'Bin':>4}  {'Pred(raw)':>10}  {'Actual(raw)':>12}  "
      f"{'Err(raw)':>9}  {'Pred(cal)':>10}  {'Actual(cal)':>12}  {'Err(cal)':>9}")
print("  " + "─" * 75)
for b in m_cal["cal_bins"]:
    print(f"  {b['bin']:>4}  {b['pred_raw']:>10.4f}  {b['actual_raw']:>12.4f}  "
          f"{b['err_raw']:>9.4f}  {b['pred_cal']:>10.4f}  {b['actual_cal']:>12.4f}  "
          f"{b['err_cal']:>9.4f}")

# ── Calibration plot ──────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# Panel 1: Calibration curve
ax1 = fig.add_subplot(gs[0])
frac_true_r = [b["actual_raw"] for b in m_cal["cal_bins"]]
frac_pred_r = [b["pred_raw"]   for b in m_cal["cal_bins"]]
frac_true_c = [b["actual_cal"] for b in m_cal["cal_bins"]]
frac_pred_c = [b["pred_cal"]   for b in m_cal["cal_bins"]]

ax1.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Perfect calibration")
ax1.plot(frac_pred_r, frac_true_r, "o-", lw=2, ms=7,
         color="#C73E1D", label=f"Raw XGBoost (MCE={m_cal['mce_raw']:.3f})")
ax1.plot(frac_pred_c, frac_true_c, "s-", lw=2, ms=7,
         color="#2E86AB", label=f"Calibrated ({CALIBRATION_METHOD}) (MCE={m_cal['mce_cal']:.3f})")
ax1.set_xlabel("Mean predicted probability", fontsize=10)
ax1.set_ylabel("Fraction of positives (actual)", fontsize=10)
ax1.set_title("Calibration Curve\n(closer to diagonal = better)", fontweight="bold", fontsize=11)
ax1.legend(fontsize=8, loc="upper left")
ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)

# Panel 2: Probability distribution
ax2 = fig.add_subplot(gs[1])
oof_r = cv_cal["oof_probs_raw"]
oof_c = cv_cal["oof_probs"]
ax2.hist(oof_r[y_best == 0], bins=20, alpha=0.4, density=True,
         color="#2E86AB", label="GO-1 High (raw)")
ax2.hist(oof_r[y_best == 1], bins=20, alpha=0.4, density=True,
         color="#C73E1D", label="GO-2 Low (raw)")
ax2.hist(oof_c[y_best == 1], bins=20, alpha=0.4, density=True,
         color="#A23B72", label="GO-2 Low (calibrated)", histtype="step",
         linewidth=2)
ax2.axvline(y_best.mean(), color="black", ls="--", lw=1.2, alpha=0.6,
            label=f"Prevalence={y_best.mean():.3f}")
ax2.set_xlabel("Predicted probability", fontsize=10)
ax2.set_ylabel("Density", fontsize=10)
ax2.set_title("Probability distributions\nRaw vs Calibrated GO-2",
              fontweight="bold", fontsize=11)
ax2.legend(fontsize=7)

# Panel 3: Brier score comparison
ax3 = fig.add_subplot(gs[2])
categories  = ["Raw\nXGBoost", f"Calibrated\n({CALIBRATION_METHOD})"]
brier_vals  = [m_cal["brier_raw"], m_cal["brier_cal"]]
colors_b    = ["#C73E1D", "#2E86AB"]
bars = ax3.bar(categories, brier_vals, color=colors_b, edgecolor="white",
               alpha=0.85, width=0.45)
ax3.axhline(y_best.mean() * (1 - y_best.mean()), color="grey", ls="--",
            lw=1, alpha=0.6, label="Naive baseline Brier")
for bar, val in zip(bars, brier_vals):
    ax3.text(bar.get_x() + bar.get_width()/2, val + 0.002,
             f"{val:.4f}", ha="center", fontsize=10, fontweight="bold")
ax3.set_ylabel("Brier Score (lower = better)", fontsize=10)
ax3.set_title(f"Brier Score\n{brier_imp:.1f}% improvement with calibration",
              fontweight="bold", fontsize=11)
ax3.legend(fontsize=8)
ax3.set_ylim(0, max(brier_vals) * 1.25)

fig.suptitle("Calibration Analysis — P1_spw\n"
             "CalibratedClassifierCV (isotonic, cv=3) applied within each CV fold",
             fontweight="bold", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Fit calibrated final model for export
print()
print("Fitting calibrated final model on full dataset...")
final_cal = fit_final_model_calibrated(
    X_best, y_best,
    params       = BEST_PARAMS,
    calib_method = CALIBRATION_METHOD,
    calib_cv     = CALIBRATION_CV,
)
print("  clf_raw : XGBClassifier         — for SHAP")
print("  clf_cal : CalibratedClassifierCV — for predict_proba in dashboard")

# Quick sanity check: p(GO-2) mean should ≈ prevalence after calibration
p_cal_train = final_cal["clf_cal"].predict_proba(final_cal["X_scaled"])[:, 1]
p_raw_train = final_cal["clf_raw"].predict_proba(final_cal["X_scaled"])[:, 1]
print()
print(f"  Prevalence GO-2        : {y_best.mean():.4f}")
print(f"  Mean p(GO-2) raw       : {p_raw_train.mean():.4f}  (may be shifted by SPW)")
print(f"  Mean p(GO-2) calibrated: {p_cal_train.mean():.4f}  (should ≈ prevalence)")


In [0]:
# ─────────────────────────────────────────────────────────────────────────
# DASHBOARD EXPORT — Calibrated model bundle
#
# Replaces paso_d_modelo_m7_best.joblib with a calibrated version.
# The bundle contains both clf_raw (SHAP) and clf_cal (production).
#
# Usage in dashboard:
#   bundle = joblib.load("kmc20_model_bundle_calibrated.joblib")
#   result = bundle["predict"](patient_dict)
#   # Returns calibrated probability — interpretable as true frequency
# ─────────────────────────────────────────────────────────────────────────
import json, joblib
import numpy as np
import shap
import datetime
from sklearn.metrics import confusion_matrix
from sklearn.calibration import calibration_curve

# ── Calibrated predict function ──────────────────────────────────────────
def make_calibrated_predict(clf_cal, imputer, scaler, feature_cols, threshold):
    """
    Returns a predict function using the calibrated classifier.
    Probabilities are interpretable: p=0.30 means ~30% of similar patients
    are GO-2 Low, independent of class imbalance artifacts.

    Risk tiers (based on clinical threshold + prevalence)
    ─────────────────────────────────────────────────────
    Alto Riesgo    : p ≥ 0.50  (clearly above threshold, high confidence)
    Riesgo Moderado: threshold ≤ p < 0.50
    Bajo Riesgo    : p < threshold
    """
    cols = feature_cols

    def predict(patient: dict) -> dict:
        row     = np.array([[patient.get(c, np.nan) for c in cols]], dtype=float)
        row_imp = imputer.transform(row)
        row_sc  = scaler.transform(row_imp)
        prob    = float(clf_cal.predict_proba(row_sc)[0, 1])

        if prob >= 0.50:
            risk_label  = "Alto Riesgo"
            alert_level = "high"
        elif prob >= threshold:
            risk_label  = "Riesgo Moderado"
            alert_level = "medium"
        else:
            risk_label  = "Bajo Riesgo"
            alert_level = "low"

        return {
            "probability"     : round(prob, 4),
            "risk_label"      : risk_label,
            "alert_level"     : alert_level,
            "above_threshold" : bool(prob >= threshold),
            "threshold_used"  : round(threshold, 3),
            "calibrated"      : True,
            "calib_method"    : CALIBRATION_METHOD,
        }

    return predict


# ── Calibrated SHAP function (uses clf_raw) ──────────────────────────────
def compute_shap_for_patient(clf_raw, imputer, scaler,
                              feature_cols, patient: dict) -> dict:
    """
    Compute per-patient SHAP values using the raw XGBoost classifier.

    CalibratedClassifierCV does not support TreeExplainer — use clf_raw.
    The SHAP direction and ranking remain valid even after calibration;
    only the magnitude of predict_proba changes.

    Returns
    -------
    dict with:
        base_prob       : float  — average model output (log-odds converted)
        shap_values     : list   — SHAP contribution per feature
        top_risk_factors: list   — features pushing toward GO-2 (positive SHAP)
        top_protective  : list   — features pushing away from GO-2 (negative SHAP)
    """
    import math
    cols    = feature_cols
    row     = np.array([[patient.get(c, np.nan) for c in cols]], dtype=float)
    row_imp = imputer.transform(row)
    row_sc  = scaler.transform(row_imp)

    explainer = shap.TreeExplainer(clf_raw)
    sv        = explainer(row_sc, check_additivity=False)
    shap_vals = sv.values[0].tolist()
    base_logo = float(sv.base_values[0])
    base_prob = 1 / (1 + math.exp(-base_logo))

    features_shap = [
        {"feature": col, "shap": round(shap_vals[i], 5)}
        for i, col in enumerate(cols)
    ]
    features_shap.sort(key=lambda x: x["shap"], reverse=True)

    return {
        "base_prob"       : round(base_prob, 4),
        "shap_values"     : [
            {"feature": col, "shap": round(shap_vals[i], 5)}
            for i, col in enumerate(cols)
        ],
        "top_risk_factors": [f for f in features_shap if f["shap"] > 0][:5],
        "top_protective"  : [f for f in features_shap if f["shap"] < 0][:3],
    }


# ── Calibrated threshold table ───────────────────────────────────────────
oof_cal   = cv_cal["oof_probs"]          # calibrated OOF probabilities
threshold = m_cal["best_threshold"]      # F1-optimal threshold on calibrated probs

threshold_rows_cal = []
for thr in np.arange(0.05, 0.96, 0.05):
    thr    = round(float(thr), 2)
    pred_t = (oof_cal >= thr).astype(int)
    cm_t   = confusion_matrix(y_best, pred_t)
    tn, fp, fn, tp = cm_t[0,0], cm_t[0,1], cm_t[1,0], cm_t[1,1]
    sens = round(float(tp/(tp+fn)) if (tp+fn)>0 else 0.0, 4)
    spec = round(float(tn/(tn+fp)) if (tn+fp)>0 else 0.0, 4)
    ppv  = round(float(tp/(tp+fp)) if (tp+fp)>0 else 0.0, 4)
    npv  = round(float(tn/(tn+fn)) if (tn+fn)>0 else 0.0, 4)
    f1_v = round(float(2*tp/(2*tp+fp+fn)) if (2*tp+fp+fn)>0 else 0.0, 4)
    threshold_rows_cal.append({
        "threshold"      : thr,
        "sensitivity"    : sens,
        "specificity"    : spec,
        "ppv"            : ppv,
        "npv"            : npv,
        "f1"             : f1_v,
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "is_recommended" : thr == round(threshold, 2),
    })

# ── SHAP reference on calibrated final model ─────────────────────────────
explainer_ref = shap.TreeExplainer(final_cal["clf_raw"])
sv_ref        = explainer_ref(final_cal["X_scaled"], check_additivity=False)
sv_ref.feature_names = cols_best
mean_abs_shap = np.abs(sv_ref.values).mean(axis=0)
base_val      = float(sv_ref.base_values[0])

shap_ref_cal = {
    "base_value"   : round(base_val, 6),
    "base_prob"    : round(float(1/(1+np.exp(-base_val))), 4),
    "note"         : "SHAP computed on clf_raw (XGBoost); calibrated prob via clf_cal",
    "features"     : [
        {
            "name"         : col,
            "mean_abs_shap": round(float(mean_abs_shap[i]), 6),
            "rank"         : int(np.argsort(mean_abs_shap)[::-1].tolist().index(i)) + 1,
        }
        for i, col in enumerate(cols_best)
    ],
}

# ── Calibration metadata ──────────────────────────────────────────────────
calib_meta = {
    "method"         : CALIBRATION_METHOD,
    "cv_folds"       : CALIBRATION_CV,
    "applied_within" : "each CV fold (no leakage)",
    "brier_raw"      : round(m_cal["brier_raw"], 5),
    "brier_cal"      : round(m_cal["brier_cal"], 5),
    "brier_improvement_pct": round(
        (m_cal["brier_raw"] - m_cal["brier_cal"]) / m_cal["brier_raw"] * 100, 1
    ),
    "mce_raw"        : m_cal["mce_raw"],
    "mce_cal"        : m_cal["mce_cal"],
    "interpretation" : (
        "Calibrated probability p=X means approximately X*100% of patients "
        "with this profile are GO-2 Low, independent of class imbalance."
    ),
    "shap_note"      : (
        "SHAP values use clf_raw (XGBClassifier). "
        "TreeExplainer does not support CalibratedClassifierCV. "
        "SHAP direction and ranking remain valid after calibration."
    ),
    "cal_bins"       : m_cal["cal_bins"],
}

# ── Build and save full bundle ─────────────────────────────────────────────
# Note: Function objects removed from bundle to avoid PicklingError
# To use in dashboard, recreate predict function:
#   predict_fn = make_calibrated_predict(bundle["clf_cal"], bundle["imputer"],
#                                         bundle["scaler"], bundle["feature_cols"],
#                                         bundle["threshold"])

bundle_cal = {
    # Inference
    "clf_raw"       : final_cal["clf_raw"],    # XGBoost — for SHAP
    "clf_cal"       : final_cal["clf_cal"],    # Calibrated — for predict_proba
    "clf"           : final_cal["clf_raw"],    # backward compat alias
    "imputer"       : final_cal["imputer"],
    "scaler"        : final_cal["scaler"],
    "feature_cols"  : cols_best,
    "threshold"     : round(threshold, 4),

    # Identity
    "run_name"      : BEST_RUN_NAME + "_calibrated",
    "model_version" : "M7_P1_spw_calibrated_v1",
    "trained_on"    : "KMC-400-20y G1 preterm n=383",
    "created_at"    : datetime.datetime.now().isoformat(),

    # Metrics
    "metrics": {
        "auc_oof"      : round(m_cal["auc_oof"], 4),
        "f1"           : round(m_cal["f1"], 4),
        "sensitivity"  : round(m_cal["sensitivity"], 4),
        "specificity"  : round(m_cal["specificity"], 4),
        "brier"        : round(m_cal["brier_cal"], 5),
        "fn"           : int(m_cal["FN"]),
        "fp"           : int(m_cal["FP"]),
        "prevalence"   : round(float(y_best.mean()), 4),
    },
    "calibration"   : calib_meta,
}

joblib.dump(bundle_cal, "kmc20_model_bundle_calibrated.joblib")
print("Saved: kmc20_model_bundle_calibrated.joblib")

# Save updated files
with open("kmc20_shap_reference_calibrated.json", "w", encoding="utf-8") as f:
    json.dump(shap_ref_cal, f, ensure_ascii=False, indent=2)
print("Saved: kmc20_shap_reference_calibrated.json")

with open("kmc20_threshold_table_calibrated.json", "w", encoding="utf-8") as f:
    json.dump(threshold_rows_cal, f, ensure_ascii=False, indent=2)
print("Saved: kmc20_threshold_table_calibrated.json")

print()
print("Bundle saved successfully.")
print("To use predict function in dashboard:")
print("  predict_fn = make_calibrated_predict(bundle['clf_cal'], bundle['imputer'],")
print("                                        bundle['scaler'], bundle['feature_cols'],")
print("                                        bundle['threshold'])")
